# CAKGI — Monte Carlo imputation benchmark (main results + ablations)

Reproduces the Monte Carlo evaluation of **CAKGI** (context-aware, knowledge-guided diffusion imputation of CGM glucose and heart rate) against MICE, BRITS, CATSI, GP-VAE and CSDI-adapted. It also reproduces the component ablations and the simple context baselines.

**Protocol (as in the paper)**
- **Patient-disjoint split.** 25 % of patients are held out (13 in our data). Every learned model is trained on the training patients only, and the normalisation statistics come from the training patients only.
- **Gaps.** 500 independent synthetic gaps per trial × 3 trials (seeds 42, 7, 123). The gaps are drawn from originally observed glucose stretches of the held-out patients, with durations of 1, 2, 3, 4, 5, 6 and 8 h. Each gap is masked and imputed on its own; all other real missingness stays visible to every method.
- **Imputation window.** Every method imputes from the same ±120-min window around the gap.
- **Metrics.** RMSE, MAE and CRPS. CRPS uses 50 samples for the probabilistic methods; for deterministic methods it equals MAE. Diffusion models are reported under median-of-chains and best-chain aggregation.
- **Training budget.** Every learned model is trained for 75 epochs on the same 48-step windows with one contiguous masked block. BRITS, GP-VAE, CSDI-adapted and CAKGI use the same LR schedule (5-epoch linear warm-up, then cosine decay). CATSI uses its official optimiser settings.
- **Statistics.** Paired differences with a patient-level cluster bootstrap (10,000 resamples of held-out patients), plus the number of patients improved and a Wilcoxon signed-rank test on per-patient mean differences.

**Baselines**

| Method | Implementation |
|---|---|
| MICE | scikit-learn `IterativeImputer` (BayesianRidge, 10 iterations) per patient |
| BRITS | Cao et al. (2018): time-since-last-observation δ, temporal decay, history and feature regression (zero diagonal), β-combination, bidirectional LSTM + consistency loss |
| GP-VAE | Fortuin et al. (2020): BiLSTM encoder on x·mask, full Cauchy-kernel GP prior, banded-precision posterior, exact KL (weight 10⁻⁴, 10-epoch KL warm-up) |
| CATSI | Official code (github.com/ai4h-hkbu/CATSI, commit 239df11), with one correction: the series mean is the mean of the observed values (paper Eq. 5) |
| CSDI-adapted | Conditional score-based diffusion (Tashiro et al., 2021) with the same window, masking and 50 chains |

**Inputs** (put them in `BASE`)
- `SDBII_gap_classified.csv`: columns `PtID`, `ts`, `GlucoseCGM`, `HR` on a regular 5-min grid, with missing samples as empty or 0. It is written by `CAKGI_run_imputation.ipynb`.
- `SDBII_context_vectors.csv`: the context vectors (with `PtID`) written by `CAKGI_run_imputation.ipynb`. Only the training patients' rows are used, and only for normalisation.

**Run order.** Run the cells top to bottom.
- **Part A** (Cells 1–15) produces Table 1, the duration tables, the figures and the CAKGI-vs-baseline statistics.
- **Part B** (Cells 16–21) produces the ablation table.

Training and evaluation are resumable: checkpoints and per-trial results are written to `BASE` and reused on a re-run. Delete a file to recompute it.

**Tables from the released results.** To rebuild the tables without retraining, run Cell 1, put the released `mc_results_all.csv` into `RESULTS_DIR`, and run Cells 13, 15 and 21.

**Reproducibility.** Seeds are fixed for data splitting, gap sampling, training and sampling. GPU kernels (CUDA/MPS) are not bit-deterministic, so retrained models and diffusion samples can differ slightly from the released numbers. In our runs, retraining the same model with a second seed changed CAKGI's glucose RMSE by 0.8 mg/dL (the "noise floor" row of the ablation table).

`SMOKE_TEST = True` (Cell 1) runs the whole notebook in a few minutes on CPU with tiny settings. It only checks that the code runs; its numbers are meaningless.

**Requirements:** Python ≥ 3.10, PyTorch ≥ 2.0 (CUDA, Apple MPS or CPU), NumPy, pandas ≥ 2.0, SciPy, scikit-learn, Matplotlib, tqdm. Tested with PyTorch 2.14, pandas 2.2 and scikit-learn 1.9.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 1 — Imports and configuration
# ════════════════════════════════════════════════════════════════════
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import gc, json, math, random, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import BayesianRidge, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import wilcoxon, binomtest
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────
BASE        = r"path/to/your/data_directory"          # <-- set this
RESULTS_DIR = f"{BASE}/mc_results"
PLOT_DIR    = f"{BASE}/mc_plots"
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(PLOT_DIR, exist_ok=True)

ORIGINAL_FILE = f"{BASE}/SDBII_gap_classified.csv"    # PtID, ts, GlucoseCGM, HR (5-min grid)
VECTORS_FILE  = f"{BASE}/SDBII_context_vectors.csv"   # written by CAKGI_run_imputation.ipynb

# ── Patient-disjoint split ────────────────────────────────────────────
SPLIT_SEED        = 2026
TEST_PATIENT_FRAC = 0.25
SPLIT_TAG         = f"mc_eligible_independent_seed{SPLIT_SEED}"
MC_SPLIT_FILE     = f"{RESULTS_DIR}/mc_patient_split_{SPLIT_TAG}.json"

# ── Monte Carlo protocol ──────────────────────────────────────────────
N_GAPS       = 500                 # independent gaps per trial
N_TRIALS     = 3
RANDOM_SEEDS = [42, 7, 123]
N_SAMPLES    = 50                  # chains / posterior samples per gap
CHAIN_BATCH_SIZE  = 50             # chains per accelerator batch (lower if memory is tight)
CACHE_CLEAR_EVERY = 50
GAP_DURATION_PROBS = {12: 0.15, 24: 0.20, 36: 0.20, 48: 0.15, 60: 0.12, 72: 0.10, 96: 0.08}  # steps (5 min)
GAP_DURATIONS = list(GAP_DURATION_PROBS.keys())
GAP_PROBS     = list(GAP_DURATION_PROBS.values())
MIN_CONTEXT   = 3

# ── Shared model / training settings ──────────────────────────────────
HYPO_THR = 70;  CTX_WIN = 120;  MAX_L = 1000
FREQ = '5min';  T_STEPS = 100;  SEQ_LEN = 48;  MASK_RATIO = 0.15
BATCH_SIZE = 32; LR = 1e-3
EPOCHS        = 75                 # ONE training budget for every learned model
WARMUP_EPOCHS = 5                  # linear LR warm-up, then cosine decay
CAUCHY_GAMMA = 0.1; CAUCHY_L = 10.0; BOUNDARY_W = 0.3
NF = 2; DM = 64; NH = 4; NL = 3
MAX_TRAIN_WINDOWS = None           # None = all training windows (smoke test only: cap)
RIDGE_TRAIN_GAPS  = 2000           # synthetic TRAIN gaps for the context-ridge baseline

# ── Smoke test: tiny settings to check that every cell runs (numbers are meaningless) ──
SMOKE_TEST = False
if SMOKE_TEST:
    N_GAPS, N_TRIALS, N_SAMPLES, CHAIN_BATCH_SIZE = 10, 2, 4, 4
    GAP_DURATION_PROBS = {12: 0.5, 24: 0.5}          # short gaps only, to keep CPU time low
    GAP_DURATIONS, GAP_PROBS = list(GAP_DURATION_PROBS), list(GAP_DURATION_PROBS.values())
    EPOCHS, WARMUP_EPOCHS = 2, 1
    MAX_TRAIN_WINDOWS, RIDGE_TRAIN_GAPS = 64, 40

# ── Checkpoints (split-specific; the names are those of the released 75-epoch models) ──
_ck = lambda name: f"{BASE}/SDBII_{name}_{SPLIT_TAG}.pt"
BRITS_MODEL    = _ck('brits_faithful_75ep')
GPVAE_MODEL    = _ck('gpvae_banded_75ep')
CATSI_MODEL    = _ck('catsi_meanfix')
CSDI_MODEL     = _ck('csdi_75ep')
CSDI_BND_MODEL = _ck('csdi_bndloss_75ep')                 # ablation
PIDCK_MODEL    = _ck('pidck_v4_model')
# A checkpoint is loaded if it exists, otherwise the model is trained and saved.
# Keep smoke-test runs in a separate BASE so tiny models never overwrite real ones.

# ── Result files ──────────────────────────────────────────────────────
MAIN_RESULTS = f"{RESULTS_DIR}/mc_results_main.csv"       # Part A (resumable per trial × method)
ABL_RESULTS  = f"{RESULTS_DIR}/mc_results_ablation.csv"   # Part B (resumable per trial × arm)
ALL_RESULTS  = f"{RESULTS_DIR}/mc_results_all.csv"        # Part A + Part B

# ── Device ────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f"Device: {DEVICE} | epochs: {EPOCHS} | gaps/trial: {N_GAPS} | samples: {N_SAMPLES}"
      + (" | SMOKE TEST" if SMOKE_TEST else ""))


def release_device_cache(force=False, gap_index=None):
    if not force and gap_index is not None and (gap_index + 1) % CACHE_CLEAR_EVERY != 0:
        return
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    elif DEVICE.type == 'mps' and hasattr(torch, 'mps'):
        try:
            torch.mps.empty_cache()
        except Exception:
            pass


def chain_chunks(total=None, batch_size=None):
    """Chain counts summing to `total` (execution batching only; the ensemble size is unchanged)."""
    total = N_SAMPLES if total is None else total
    batch_size = CHAIN_BATCH_SIZE if batch_size is None else batch_size
    done = 0
    while done < total:
        n = min(batch_size, total - done)
        yield n
        done += n


MAIN_METHODS  = ['mice', 'brits', 'catsi', 'gpvae', 'csdi', 'pidck']
METHOD_LABELS = {'mice': 'MICE', 'brits': 'BRITS', 'catsi': 'CATSI', 'gpvae': 'GP-VAE',
                 'csdi': 'CSDI-adapted', 'pidck': 'CAKGI'}
COLORS = {'mice': '#4C72B0', 'brits': '#55A868', 'catsi': '#DD8452', 'gpvae': '#C44E52',
          'csdi': '#8C8C8C', 'pidck': '#FF6600'}

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 2 — Load data, eligibility-aware patient-disjoint split, train-only normalisation
# ════════════════════════════════════════════════════════════════════
df_orig = pd.read_csv(ORIGINAL_FILE, low_memory=False)
df_orig['ts'] = pd.to_datetime(df_orig['ts'], format='mixed', dayfirst=True)
df_orig = df_orig.sort_values(['PtID','ts']).reset_index(drop=True)
df_orig['GlucoseCGM'] = df_orig['GlucoseCGM'].replace(0, np.nan)
df_orig['HR']         = df_orig['HR'].replace(0, np.nan)

ALL_PIDS = np.array(sorted(df_orig['PtID'].dropna().unique().tolist(), key=str), dtype=object)


def _has_observed_gap_candidate(pat, gap_len, min_context=MIN_CONTEXT):
    """
    Patient-level MC eligibility check using GLUCOSE only.

    A candidate requires:
      - the complete synthetic target interval to be originally observed;
      - one originally observed glucose value immediately before and after;
      - enough indexing room for the configured minimum context.

    Eligibility uses only missingness/observability, never outcome performance.
    """
    pat = pat.sort_values('ts').reset_index(drop=True)
    g = pat['GlucoseCGM'].to_numpy(dtype=float)
    n = len(g)
    first = max(int(min_context), 1)
    last_exclusive = n - int(gap_len) - max(int(min_context), 1) + 1
    if last_exclusive <= first:
        return False

    starts = np.arange(first, last_exclusive, dtype=np.int32)
    miss = (~np.isfinite(g)).astype(np.int32)
    cs = np.concatenate(([0], np.cumsum(miss, dtype=np.int64)))
    gap_missing = cs[starts + gap_len] - cs[starts]
    pre_ok = np.isfinite(g[starts - 1])
    post_ok = np.isfinite(g[starts + gap_len])
    return bool(np.any((gap_missing == 0) & pre_ok & post_ok))


# --------------------------------------------------------------------
# Choose MC test patients from patients that can actually support the
# requested long-gap benchmark. This avoids a random holdout in which
# most patients can never contribute a Monte Carlo sample.
#
# IMPORTANT: after TEST_PIDS are chosen, ALL learned models are trained
# (or loaded from split-specific checkpoints) on the remaining TRAIN_PIDS
# only. Hence there is no train/test patient overlap.
# --------------------------------------------------------------------
longest_gap = int(max(GAP_DURATIONS))
shortest_gap = int(min(GAP_DURATIONS))
long_eligible = []
short_eligible = []

for pid, pat in df_orig.groupby('PtID'):
    if _has_observed_gap_candidate(pat, longest_gap):
        long_eligible.append(pid)
        short_eligible.append(pid)
    elif _has_observed_gap_candidate(pat, shortest_gap):
        short_eligible.append(pid)

long_eligible = np.array(sorted(long_eligible, key=str), dtype=object)
short_eligible = np.array(sorted(set(short_eligible), key=str), dtype=object)

rng_split = np.random.default_rng(SPLIT_SEED)
n_test_target = max(1, int(np.ceil(TEST_PATIENT_FRAC * len(ALL_PIDS))))

# Prefer patients with at least one valid 8-hour candidate; if there are
# fewer than the target count, fill from patients that support >=1 hour.
if len(long_eligible) >= n_test_target:
    TEST_PIDS = rng_split.choice(long_eligible, size=n_test_target, replace=False)
else:
    chosen = list(long_eligible)
    remaining = np.array([p for p in short_eligible if p not in set(chosen)], dtype=object)
    need = n_test_target - len(chosen)
    if need > 0 and len(remaining) > 0:
        take = min(need, len(remaining))
        chosen.extend(rng_split.choice(remaining, size=take, replace=False).tolist())
    TEST_PIDS = np.array(chosen, dtype=object)

if len(TEST_PIDS) == 0:
    raise RuntimeError(
        "No patient has an originally observed interval suitable for the Monte Carlo benchmark."
    )

if len(TEST_PIDS) < n_test_target:
    print(
        f"WARNING: requested {n_test_target} MC test patients but only "
        f"{len(TEST_PIDS)} patients satisfy the observability eligibility rule."
    )

TEST_SET = set(TEST_PIDS.tolist())
TRAIN_PIDS = np.array([p for p in ALL_PIDS if p not in TEST_SET], dtype=object)

train_df = df_orig[df_orig['PtID'].isin(TRAIN_PIDS)].copy().reset_index(drop=True)
test_df  = df_orig[df_orig['PtID'].isin(TEST_PIDS)].copy().reset_index(drop=True)

assert set(map(str, TRAIN_PIDS)).isdisjoint(set(map(str, TEST_PIDS)))
assert len(train_df) + len(test_df) == len(df_orig)

# Fit normalization ONLY on training patients.
G_MEAN = train_df['GlucoseCGM'].mean(); G_STD = train_df['GlucoseCGM'].std()
H_MEAN = train_df['HR'].mean();          H_STD = train_df['HR'].std()
G_STD  = G_STD if pd.notna(G_STD) and G_STD > 0 else 1.0
H_STD  = H_STD if pd.notna(H_STD) and H_STD > 0 else 1.0

# Persist the exact split (SPLIT_TAG is part of every checkpoint name).
split_payload = {
    'split_seed': int(SPLIT_SEED),
    'split_tag': SPLIT_TAG,
    'eligibility_rule': (
        'MC test patients selected using signal observability only; prefer patients '
        'with >=1 originally observed 8h glucose interval plus observed immediate boundaries'
    ),
    'target_test_patients': int(n_test_target),
    'long_gap_eligible_patients_total': int(len(long_eligible)),
    'short_gap_eligible_patients_total': int(len(short_eligible)),
    'train_pids': [str(x) for x in TRAIN_PIDS],
    'test_pids': [str(x) for x in TEST_PIDS],
}
with open(MC_SPLIT_FILE, 'w') as f:
    json.dump(split_payload, f, indent=2)

print(f"Loaded {len(df_orig):,} rows | {len(ALL_PIDS)} patients")
print(f"MC eligibility: {len(long_eligible)} patients support >=8h; "
      f"{len(short_eligible)} support >=1h")
print(f"TRAIN: {len(TRAIN_PIDS)} patients, {len(train_df):,} rows")
print(f"TEST : {len(TEST_PIDS)} MC-eligible patients, {len(test_df):,} rows")
print(f"Train-only stats | Glucose mean={G_MEAN:.2f} std={G_STD:.2f} | "
      f"HR mean={H_MEAN:.2f} std={H_STD:.2f}")
print(f"Split manifest: {MC_SPLIT_FILE}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 3 — Independent, duration-balanced Monte Carlo gap sampler
#
# Each Monte Carlo gap is an independent evaluation sample:
#   - different samples MAY overlap in the original signal;
#   - exact duplicate (patient, start, duration) samples are not reused;
#   - when a sample is evaluated, ONLY that sample's target interval is masked.
# ════════════════════════════════════════════════════════════════════

def _build_patient_cache(df):
    return {
        pid: pat.sort_values('ts').reset_index(drop=True)
        for pid, pat in df.groupby('PtID')
    }


def _duration_quotas(n_gaps, gap_durations, gap_probs):
    """Largest-remainder integer quotas summing exactly to n_gaps."""
    gap_durations = [int(x) for x in gap_durations]
    probs = np.asarray(gap_probs, dtype=float)
    probs = probs / probs.sum()
    expected = probs * int(n_gaps)
    base = np.floor(expected).astype(int)
    rem = int(n_gaps) - int(base.sum())
    if rem > 0:
        order = np.argsort(-(expected - base))
        for j in order[:rem]:
            base[j] += 1
    return {gl: int(q) for gl, q in zip(gap_durations, base)}


def _valid_starts_for_patient(pat, gap_len, min_context=MIN_CONTEXT):
    """
    Candidate target interval must be fully observed in glucose and have
    observed immediate pre/post glucose boundaries. Existing missingness in
    wider context is allowed and remains visible to every imputer.
    """
    g = pat['GlucoseCGM'].to_numpy(dtype=float)
    n = len(g)
    first = max(int(min_context), 1)
    last_exclusive = n - int(gap_len) - max(int(min_context), 1) + 1
    if last_exclusive <= first:
        return np.empty(0, dtype=np.int32)

    starts = np.arange(first, last_exclusive, dtype=np.int32)
    miss = (~np.isfinite(g)).astype(np.int32)
    cs = np.concatenate(([0], np.cumsum(miss, dtype=np.int64)))
    gap_missing = cs[starts + gap_len] - cs[starts]
    pre_ok = np.isfinite(g[starts - 1])
    post_ok = np.isfinite(g[starts + gap_len])
    return starts[(gap_missing == 0) & pre_ok & post_ok]


def _prepare_candidate_cache(patient_cache, gap_durations, min_context=MIN_CONTEXT):
    cache = {}
    print("  Precomputing valid independent gap positions...", flush=True)
    for gap_len in gap_durations:
        gap_len = int(gap_len)
        per_pid = {}
        total = 0
        for pid, pat in patient_cache.items():
            starts = _valid_starts_for_patient(pat, gap_len, min_context)
            if len(starts):
                per_pid[pid] = starts
                total += len(starts)
        cache[gap_len] = per_pid
        print(
            f"    {gap_len*5:>3} min: {total:,} unique starts across "
            f"{len(per_pid)}/{len(patient_cache)} patients",
            flush=True
        )
    return cache


def _make_gap(patient_cache, pid, idx, gap_len):
    pat = patient_cache[pid]
    ts_arr = pat['ts'].values[idx:idx + gap_len]
    gt_g = pat['GlucoseCGM'].values[idx:idx + gap_len]
    gt_h = pat['HR'].values[idx:idx + gap_len]
    return {
        'pid': pid,
        'start_idx': int(idx),
        'gap_len': int(gap_len),
        'gap_len_min': int(gap_len) * 5,
        'ts_start': pd.Timestamp(ts_arr[0]),
        'ts_end': pd.Timestamp(ts_arr[-1]),
        'gt_glucose': [float(v) for v in gt_g],
        'gt_hr': [float(v) if np.isfinite(v) else np.nan for v in gt_h],
    }


def simulate_gaps(df, n_gaps, gap_durations, gap_probs, rng,
                  min_context=MIN_CONTEXT):
    """
    Select exactly n_gaps independent Monte Carlo samples with fixed duration
    quotas and balanced patient contribution.

    Overlap BETWEEN different Monte Carlo samples is allowed because each is
    later masked/evaluated independently. Exact duplicate samples are avoided.
    """
    import time
    t0 = time.perf_counter()
    patient_cache = _build_patient_cache(df)
    candidate_cache = _prepare_candidate_cache(
        patient_cache, gap_durations, min_context
    )
    quotas = _duration_quotas(n_gaps, gap_durations, gap_probs)

    print("  Target duration quotas: " + " | ".join(
        f"{gl*5}min={quotas[gl]}" for gl in sorted(quotas)
    ), flush=True)

    selected = []
    total_counts = {pid: 0 for pid in patient_cache}
    per_duration_counts = {
        gl: {pid: 0 for pid in patient_cache} for gl in quotas
    }

    # Randomized, without-replacement start pools for every patient/duration.
    pools = {}
    cursors = {}
    for gl, per_pid in candidate_cache.items():
        pools[gl] = {}
        cursors[gl] = {}
        for pid, starts in per_pid.items():
            pools[gl][pid] = rng.permutation(starts)
            cursors[gl][pid] = 0

    # Longest-first only affects ordering/progress. It no longer affects
    # feasibility because samples are independent and may overlap.
    for gl in sorted(quotas, reverse=True):
        target = quotas[gl]
        for _ in range(target):
            available = [
                pid for pid in pools.get(gl, {})
                if cursors[gl][pid] < len(pools[gl][pid])
            ]
            if not available:
                raise RuntimeError(
                    f"Unable to draw {target} unique independent samples of "
                    f"{gl*5} min from the selected patients."
                )

            # First balance THIS duration across eligible patients, then use
            # total counts as a secondary tie-breaker.
            min_d = min(per_duration_counts[gl][pid] for pid in available)
            c1 = [pid for pid in available
                  if per_duration_counts[gl][pid] == min_d]
            min_total = min(total_counts[pid] for pid in c1)
            c2 = [pid for pid in c1 if total_counts[pid] == min_total]
            pid = c2[int(rng.integers(0, len(c2)))]

            cur = cursors[gl][pid]
            idx = int(pools[gl][pid][cur])
            cursors[gl][pid] = cur + 1

            selected.append(_make_gap(patient_cache, pid, idx, gl))
            per_duration_counts[gl][pid] += 1
            total_counts[pid] += 1

        print(f"    selected {gl*5:>3} min: {target}/{target}", flush=True)

    # Shuffle final order so downstream method timing is not grouped by duration.
    order = rng.permutation(len(selected))
    selected = [selected[int(i)] for i in order]

    elapsed = (time.perf_counter() - t0) / 60.0
    contributing = [pid for pid, c in total_counts.items() if c > 0]
    counts = np.array([total_counts[pid] for pid in contributing], dtype=int)

    print(f"  Gap sampling finished in {elapsed:.2f} min", flush=True)
    print(
        f"  {len(selected)}/{n_gaps} independent MC gaps | "
        f"{len(contributing)}/{len(patient_cache)} patients contributed",
        flush=True
    )
    print("  Achieved quotas: " + " | ".join(
        f"{gl*5}min={sum(1 for g in selected if g['gap_len']==gl)}"
        for gl in sorted(quotas)
    ), flush=True)
    if len(counts):
        print(
            f"  Gaps/patient: min={counts.min()}, "
            f"median={np.median(counts):.1f}, max={counts.max()}",
            flush=True
        )

    assert len(selected) == n_gaps
    return df, selected  # df remains UNMASKED; each sample is masked independently later

In [ ]:
# ====================================================================
# CELL 4 — Evaluation metrics: RMSE, MAE, CRPS
# ====================================================================
def empirical_crps(gt, samples):
    """
    Empirical CRPS averaged over target timesteps.

    Parameters
    ----------
    gt : array-like, shape (T,)
        Ground-truth values in original physical units.
    samples : array-like, shape (M, T) or (T,)
        Predictive ensemble in the same units. A single point prediction
        is treated as M=1 (a degenerate predictive distribution).

    Returns
    -------
    float
        Mean CRPS over valid target timesteps. Lower is better.

    For an empirical ensemble {x_m}, CRPS at one target y is:
        mean_m |x_m - y| - (1 / (2 M^2)) sum_m sum_n |x_m - x_n|.

    The pairwise term is computed with the sorted-sample identity to avoid
    constructing an M x M matrix at every timestep.
    """
    y = np.asarray(gt, dtype=np.float64).reshape(-1)
    s = np.asarray(samples, dtype=np.float64)
    if s.ndim == 1:
        s = s[None, :]
    if s.ndim != 2:
        raise ValueError(f"samples must have shape (M,T) or (T,), got {s.shape}")
    if s.shape[1] != len(y):
        raise ValueError(f"CRPS length mismatch: gt={len(y)}, samples={s.shape}")

    scores = []
    for t, yt in enumerate(y):
        if not np.isfinite(yt):
            continue
        vals = s[:, t]
        vals = vals[np.isfinite(vals)]
        M = len(vals)
        if M == 0:
            continue

        vals = np.sort(vals)
        term_obs = np.mean(np.abs(vals - yt))

        # 1/(2M^2) * sum_{i,j}|x_i-x_j|
        # = 1/M^2 * sum_i (2i-M-1) x_(i), i=1..M
        coeff = 2.0 * np.arange(1, M + 1) - M - 1.0
        term_ens = np.sum(coeff * vals) / (M * M)

        scores.append(term_obs - term_ens)

    return float(np.mean(scores)) if scores else np.nan


def compute_metrics(gt, pred, samples=None):
    """
    Compute the ONLY three reported metrics: RMSE, MAE, CRPS.

    RMSE/MAE use `pred`.
    CRPS uses `samples` when available; otherwise `pred` is treated as a
    one-member/degenerate predictive distribution (CRPS == MAE).
    """
    gt = np.asarray(gt, dtype=np.float64).reshape(-1)
    pred = np.asarray(pred, dtype=np.float64).reshape(-1)

    valid = np.isfinite(gt) & np.isfinite(pred)
    if valid.sum() == 0:
        return {'rmse': np.nan, 'mae': np.nan, 'crps': np.nan}

    gt_v = gt[valid]
    pred_v = pred[valid]
    rmse = float(np.sqrt(np.mean((gt_v - pred_v) ** 2)))
    mae = float(np.mean(np.abs(gt_v - pred_v)))

    if samples is None:
        crps = empirical_crps(gt, pred)
    else:
        crps = empirical_crps(gt, samples)

    return {'rmse': rmse, 'mae': mae, 'crps': crps}


# Sanity check: for a deterministic point forecast, CRPS must equal MAE.
_test_gt = np.array([0.0, 2.0])
_test_pred = np.array([1.0, 1.0])
assert np.isclose(empirical_crps(_test_gt, _test_pred),
                  np.mean(np.abs(_test_gt - _test_pred)))
print("Metrics ready: RMSE, MAE, CRPS")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 5 — Shared helpers: training windows, LR schedule, chain selection, MICE
# ════════════════════════════════════════════════════════════════════
def lr_schedule(opt, n_epochs=None, warmup=None):
    """The LR schedule shared by BRITS, GP-VAE, CSDI-adapted and CAKGI:
    linear warm-up for `warmup` epochs, then cosine decay. Call the returned
    function once at the end of every epoch."""
    n_epochs = EPOCHS if n_epochs is None else n_epochs
    warmup = WARMUP_EPOCHS if warmup is None else warmup
    warm = torch.optim.lr_scheduler.LambdaLR(opt, lambda ep: min(1.0, (ep + 1) / warmup))
    cos  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, max(1, n_epochs - warmup))
    def step(ep):
        (warm if ep < warmup else cos).step()
    return step


def log_epoch(ep):
    return ep == 0 or (ep + 1) % 10 == 0 or (ep + 1) == EPOCHS


def limit_windows(samples, seed=SPLIT_SEED):
    """Smoke test only: keep a random subset of training windows (no-op when MAX_TRAIN_WINDOWS is None)."""
    if MAX_TRAIN_WINDOWS is None or len(samples) <= MAX_TRAIN_WINDOWS:
        return samples
    keep = np.random.default_rng(seed).choice(len(samples), MAX_TRAIN_WINDOWS, replace=False)
    return [samples[i] for i in sorted(keep)]


class CGMDatasetSeq(Dataset):
    """Training windows for BRITS, GP-VAE and CATSI: fully observed 48-step (4-h) windows
    from TRAIN patients (50 % overlap), one contiguous block of 15 % of the steps masked."""
    def __init__(self, df, seq=SEQ_LEN, mr=MASK_RATIO):
        self.s = []
        for pid, p in df.groupby('PtID'):
            p = p.sort_values('ts').reset_index(drop=True)
            g = (p['GlucoseCGM'].values - G_MEAN) / G_STD
            h = (p['HR'].values - H_MEAN) / H_STD
            sig = np.stack([g, h], axis=1)
            for i in range(0, len(sig) - seq, seq // 2):
                w = sig[i:i+seq]
                if np.isnan(w).any():
                    continue
                self.s.append(w.copy())
        self.s = limit_windows(self.s)
    def __len__(self):
        return len(self.s)
    def __getitem__(self, i):
        x = torch.tensor(self.s[i], dtype=torch.float32)       # full clean window
        m = torch.ones_like(x)
        nm = max(1, int(SEQ_LEN * MASK_RATIO))
        st = int(torch.randint(0, SEQ_LEN - nm + 1, (1,)).item())
        m[st:st+nm] = 0                                       # contiguous masked block
        return x, m


def select_best_chain(samps, x_init_np, mask_np, target_start=None, target_len=None):
    """
    Best-chain protocol: the chain whose values at the first/last target step are closest
    to the observed values immediately before/after the SYNTHETIC target gap.
    Natural missingness elsewhere in the window is ignored for the selection.

    samps : (N, L, NF) or (N, NF, L); x_init_np, mask_np : (L, NF) or (NF, L).
    """
    N = samps.shape[0]
    scores = np.zeros(N, dtype=np.float64)
    if samps.ndim == 3 and samps.shape[2] == NF:
        s = samps.transpose(0, 2, 1)
        xi = x_init_np.T if x_init_np.ndim == 2 else x_init_np
        mk = mask_np.T if mask_np.ndim == 2 else mask_np
    else:
        s = samps
        xi = x_init_np.T if (x_init_np.ndim == 2 and x_init_np.shape[1] == NF) else x_init_np
        mk = mask_np.T if (mask_np.ndim == 2 and mask_np.shape[1] == NF) else mask_np
    L = s.shape[-1]
    if target_start is None or target_len is None:
        gap_by_feature = [np.where(mk[f] == 0)[0] for f in range(s.shape[1])]
    else:
        a = max(0, int(target_start)); b = min(L, a + int(target_len))
        gap_by_feature = [np.arange(a, b, dtype=int) for _ in range(s.shape[1])]
    for f in range(s.shape[1]):
        gap = gap_by_feature[f]
        if len(gap) == 0:
            continue
        obs = np.where(mk[f] > 0)[0]
        pre_obs = obs[obs < gap[0]]; post_obs = obs[obs > gap[-1]]
        if len(pre_obs) == 0 and len(post_obs) == 0:
            continue
        if len(pre_obs) > 0:
            scores += np.abs(s[:, f, gap[0]].astype(float) - float(xi[f, pre_obs[-1]]))
        if len(post_obs) > 0:
            scores += np.abs(s[:, f, gap[-1]].astype(float) - float(xi[f, post_obs[0]]))
    return int(np.argmin(scores))


# ── MICE ──────────────────────────────────────────────────────────────
def impute_mice(masked_df, gap_info):
    results = {id(g): {'glucose': [np.nan]*g['gap_len'], 'hr': [np.nan]*g['gap_len']}
               for g in gap_info}
    imp = IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42, verbose=0)
    pid_gaps = {}
    for g in gap_info:
        pid_gaps.setdefault(g['pid'], []).append(g)
    for pid, gaps in pid_gaps.items():
        pat = masked_df[masked_df['PtID'] == pid].sort_values('ts').reset_index(drop=True)
        X = pat[['GlucoseCGM', 'HR']].values.copy().astype(np.float64)
        try:
            Xi = imp.fit_transform(X)
        except Exception as e:
            print(f"  MICE failed for {pid}: {e}"); continue
        for g in gaps:
            idx = g['start_idx']; gl = g['gap_len']
            results[id(g)] = {'glucose': Xi[idx:idx+gl, 0].tolist(), 'hr': Xi[idx:idx+gl, 1].tolist()}
    return results

print("Shared helpers ready.")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 6 — BRITS (Cao et al., NeurIPS 2018)
#   1. Time since the last observation δ_t, per feature, in hours; grows across a missing stretch.
#   2. Temporal decay of the hidden state (γ_h) and a diagonal input decay (γ_x).
#   3. History regression x̂_t from the decayed hidden state; complement x^c_t.
#   4. Feature regression ẑ_t from x^c_t with a ZERO diagonal (each signal from the other one).
#   5. Combination weight β_t from [γ_x, m_t]; ĉ_t = β ẑ + (1 − β) x̂.
#   6. LSTM cell on [c^c_t, m_t]; forward and backward directions averaged.
#   7. Loss: the BRITS L1 terms (x̂, ẑ, ĉ) on observed positions PLUS the same terms on the
#      artificially masked block (the task-matched target used by every other method),
#      and the forward/backward consistency term (× 0.1).
#   Hidden size 64; 75 epochs with the shared LR schedule; deterministic (CRPS = MAE).
# ════════════════════════════════════════════════════════════════════
BRITS_HID  = 64
STEP_HOURS = 5.0 / 60.0          # one 5-min step; δ is measured in hours


class TemporalDecay(nn.Module):
    """γ = exp(−max(0, W δ + b)); W diagonal for the input decay (Cao et al., 2018)."""
    def __init__(self, d_in, d_out, diag=False):
        super().__init__()
        self.diag = diag
        self.W = nn.Parameter(torch.empty(d_out, d_in))
        self.b = nn.Parameter(torch.empty(d_out))
        s = 1.0 / np.sqrt(d_out)
        nn.init.uniform_(self.W, -s, s); nn.init.uniform_(self.b, -s, s)
        if diag:
            assert d_in == d_out
            self.register_buffer('m', torch.eye(d_in))

    def forward(self, d):
        W = self.W * self.m if self.diag else self.W
        return torch.exp(-F.relu(F.linear(d, W, self.b)))


class FeatureRegression(nn.Module):
    """ẑ = W x + b with a zero diagonal: each feature is estimated from the others only."""
    def __init__(self, nf):
        super().__init__()
        self.W = nn.Parameter(torch.empty(nf, nf))
        self.b = nn.Parameter(torch.empty(nf))
        s = 1.0 / np.sqrt(nf)
        nn.init.uniform_(self.W, -s, s); nn.init.uniform_(self.b, -s, s)
        self.register_buffer('m', 1.0 - torch.eye(nf))

    def forward(self, x):
        return F.linear(x, self.W * self.m, self.b)


class RITSFaithful(nn.Module):
    def __init__(self, nf=NF, hidden=BRITS_HID):
        super().__init__()
        self.h = hidden
        self.temp_decay_h   = TemporalDecay(nf, hidden)
        self.temp_decay_x   = TemporalDecay(nf, nf, diag=True)
        self.hist_reg       = nn.Linear(hidden, nf)
        self.feat_reg       = FeatureRegression(nf)
        self.weight_combine = nn.Linear(nf * 2, nf)
        self.rnn_cell       = nn.LSTMCell(nf * 2, hidden)

    @staticmethod
    def deltas(mask):
        """δ_t = Δs + (1 − m_{t−1}) δ_{t−1}, δ_0 = 0 (per feature, in hours)."""
        B, L, nf = mask.shape
        d = [torch.zeros(B, nf, device=mask.device, dtype=mask.dtype)]
        for t in range(1, L):
            d.append(STEP_HOURS + (1 - mask[:, t-1]) * d[-1])
        return torch.stack(d, dim=1)

    def forward(self, x, mask):
        """x, mask: (B, L, NF); x is 0 where mask is 0.
        Returns x̂, ẑ, ĉ (estimates) and c^c (imputed series), each (B, L, NF)."""
        B, L, nf = x.shape
        d = self.deltas(mask)
        h = torch.zeros(B, self.h, device=x.device, dtype=x.dtype)
        c = torch.zeros(B, self.h, device=x.device, dtype=x.dtype)
        xs, zs, cs, imps = [], [], [], []
        for t in range(L):
            xt, mt, dt = x[:, t], mask[:, t], d[:, t]
            gamma_h = self.temp_decay_h(dt)
            gamma_x = self.temp_decay_x(dt)
            h = h * gamma_h
            x_h = self.hist_reg(h)
            x_c = mt * xt + (1 - mt) * x_h
            z_h = self.feat_reg(x_c)
            beta = torch.sigmoid(self.weight_combine(torch.cat([gamma_x, mt], dim=-1)))
            c_h = beta * z_h + (1 - beta) * x_h
            c_c = mt * xt + (1 - mt) * c_h
            h, c = self.rnn_cell(torch.cat([c_c, mt], dim=-1), (h, c))
            xs.append(x_h); zs.append(z_h); cs.append(c_h); imps.append(c_c)
        st = lambda v: torch.stack(v, dim=1)
        return st(xs), st(zs), st(cs), st(imps)


class BRITSFaithful(nn.Module):
    """Bidirectional RITS (Cao et al., 2018). forward() returns the imputed series,
    so the existing impute_brits() is reused unchanged."""
    def __init__(self, nf=NF, hidden=BRITS_HID):
        super().__init__()
        self.fwd = RITSFaithful(nf, hidden)
        self.bwd = RITSFaithful(nf, hidden)

    def _both(self, x, mask):
        f = self.fwd(x, mask)
        b = self.bwd(torch.flip(x, [1]), torch.flip(mask, [1]))
        b = tuple(torch.flip(v, [1]) for v in b)
        return f, b

    def forward(self, x, mask):
        x = x * mask
        (_, _, _, imp_f), (_, _, _, imp_b) = self._both(x, mask)
        return (imp_f + imp_b) / 2

    @staticmethod
    def _l1(est, x, w):
        return ((est - x).abs() * w).sum() / (w.sum() + 1e-5)

    def loss(self, x, mask):
        """x: FULL clean window; mask: 1 = observed, 0 = artificially masked block."""
        f, b = self._both(x * mask, mask)              # the model never sees the masked values
        tgt = 1 - mask
        total = 0.0
        for (x_h, z_h, c_h, _) in (f, b):
            for est in (x_h, z_h, c_h):
                total = total + self._l1(est, x, mask) + self._l1(est, x, tgt)
        total = total + 0.1 * (f[3] - b[3]).abs().mean()   # consistency (Cao et al., 2018)
        return total


def train_brits_faithful(df, path=BRITS_MODEL):
    model = BRITSFaithful().to(DEVICE)
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=DEVICE)); model.eval()
        print(f"  BRITS (faithful) loaded: {os.path.basename(path)}")
        return model
    np.random.seed(SPLIT_SEED); torch.manual_seed(SPLIT_SEED)
    ld  = DataLoader(CGMDatasetSeq(df), batch_size=BATCH_SIZE, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = lr_schedule(opt)
    model.train()
    for ep in range(EPOCHS):
        el = 0.0
        for x, m in ld:
            x, m = x.to(DEVICE), m.to(DEVICE)
            opt.zero_grad(); l = model.loss(x, m); l.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.); opt.step(); el += l.item()
        sched(ep)
        if log_epoch(ep):
            print(f"    BRITS (faithful) Ep{ep+1}/{EPOCHS} | {el/len(ld):.4f}", flush=True)
    model.eval()
    torch.save(model.state_dict(), path)
    print(f"  saved: {path}")
    return model


def impute_brits(masked_df,gap_info,model):
    results={}; model.eval()
    for g in gap_info:
        pid=g['pid']; gl=g['gap_len']
        pat=masked_df[masked_df['PtID']==pid].sort_values('ts').reset_index(drop=True)
        idx=g['start_idx']; ws=max(0,idx-CTX_WIN//5); we=min(len(pat),idx+gl+CTX_WIN//5)
        seg=pat.iloc[ws:we]
        gn=(seg['GlucoseCGM'].values-G_MEAN)/G_STD
        hn=(seg['HR'].values-H_MEAN)/H_STD
        sig=np.stack([gn,hn],axis=1)
        msk=(~np.isnan(sig)).astype(np.float32)
        sig=np.nan_to_num(sig,nan=0.)   # 0 at missing for GRU input
        x=torch.tensor(sig,dtype=torch.float32).unsqueeze(0).to(DEVICE)
        m=torch.tensor(msk,dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.inference_mode(): out=model(x,m).squeeze(0).cpu().numpy()
        li=idx-ws
        results[id(g)]={
            'glucose':(out[li:li+gl,0]*G_STD+G_MEAN).tolist(),
            'hr':     (out[li:li+gl,1]*H_STD+H_MEAN).tolist()}
    return results

print("BRITS ready.")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 7 — GP-VAE (Fortuin et al., AISTATS 2020)
#   1. BiLSTM encoder on [x·mask, mask]: the hidden values are never seen.
#   2. Full L×L Cauchy-kernel GP prior per latent dimension (σ = 1, length scale 4).
#   3. Structured posterior with banded (tridiagonal) precision Λ = BᵀB, B upper-bidiagonal,
#      so posterior samples are correlated over time; |B_l,l+1| ≤ 0.95·B_l,l for stability.
#   4. KL( N(μ, Λ⁻¹) || N(0, K) ) computed exactly; KL weight 1e-4, ramped up linearly over
#      the first 10 epochs.
#   MLP decoder; 75 epochs with the shared LR schedule. Point estimate = decoded posterior
#   mean; CRPS uses N_SAMPLES posterior draws.
# ════════════════════════════════════════════════════════════════════
GPVAE_KL_WEIGHT  = 1e-4
KL_WARMUP_EPOCHS = 10 if not SMOKE_TEST else 1


class GPVAEBanded(nn.Module):
    """GP-VAE: BiLSTM encoder, MLP decoder, full Cauchy-kernel GP prior over time,
    posterior with banded precision Λ = BᵀB (B upper-bidiagonal) per latent dimension."""
    def __init__(self, n_features=NF, latent=32, hidden=64,
                 gp_sigma=1.0, gp_length=4.0, kl_weight=0.01, jitter=1e-3, max_ratio=0.95):
        super().__init__()
        self.n, self.lat = n_features, latent
        self.sigma, self.length, self.kl_w, self.jitter = gp_sigma, gp_length, kl_weight, jitter
        self.max_ratio = max_ratio
        self.enc_lstm = nn.LSTM(n_features * 2, hidden, num_layers=2,
                                batch_first=True, bidirectional=True)
        self.enc_mu   = nn.Linear(hidden * 2, latent)
        self.enc_diag = nn.Linear(hidden * 2, latent)    # B_l,l   (> 0)
        self.enc_off  = nn.Linear(hidden * 2, latent)    # B_l,l+1 (last step unused)
        self.dec = nn.Sequential(nn.Linear(latent, hidden), nn.GELU(),
                                 nn.Linear(hidden, hidden), nn.GELU(),
                                 nn.Linear(hidden, n_features))
        self._prior_cache = {}

    def _prior(self, L, device):
        """K⁻¹ and log|K| for the L×L Cauchy kernel (float64 Cholesky on CPU, cached per L)."""
        key = (L, str(device))
        if key not in self._prior_cache:
            pos = torch.arange(L, dtype=torch.float64)
            K = self.sigma**2 / (1 + (pos[None] - pos[:, None])**2 / self.length**2)
            K = K + self.jitter * torch.eye(L, dtype=torch.float64)
            C = torch.linalg.cholesky(K)
            self._prior_cache[key] = (torch.cholesky_inverse(C).float().to(device),
                                      float(2 * torch.log(torch.diagonal(C)).sum()))
        return self._prior_cache[key]

    def encode(self, x, mask):
        h, _ = self.enc_lstm(torch.cat([x * mask, mask], dim=-1))    # hidden values never seen
        mu     = self.enc_mu(h)                                        # (B, L, D)
        b_diag = F.softplus(self.enc_diag(h)) + 1e-3                   # (B, L, D)
        b_off  = self.max_ratio * torch.tanh(self.enc_off(h)) * b_diag
        return mu, b_diag, b_off

    @staticmethod
    def binv(b_diag, b_off):
        """X = B⁻¹ for upper-bidiagonal B (diag b_diag, super-diag b_off[:, :-1]),
        by back-substitution; device-safe (no triangular solver). Returns (B, D, L, L)."""
        Bsz, L, D = b_diag.shape
        bd, bo = b_diag.permute(0, 2, 1), b_off.permute(0, 2, 1)        # (B, D, L)
        I = torch.eye(L, device=b_diag.device, dtype=b_diag.dtype)
        rows = [None] * L
        rows[L-1] = I[L-1].expand(Bsz, D, L) / bd[..., L-1:L]
        for l in range(L - 2, -1, -1):
            rows[l] = (I[l] - bo[..., l:l+1] * rows[l+1]) / bd[..., l:l+1]
        return torch.stack(rows, dim=-2)                                 # row l = X[l, :]

    def sample_z(self, mu, X, n=None):
        """z = μ + X ε, Cov = X Xᵀ = Λ⁻¹. mu: (B, L, D), X: (B, D, L, L)."""
        if n is None:
            eps = torch.randn(mu.shape[0], mu.shape[2], mu.shape[1], device=mu.device, dtype=mu.dtype)
            return mu + torch.einsum('bdlm,bdm->bld', X, eps)
        eps = torch.randn(n, mu.shape[2], mu.shape[1], device=mu.device, dtype=mu.dtype)
        return mu[0].unsqueeze(0) + torch.einsum('dlm,ndm->nld', X[0], eps)   # (n, L, D)

    def kl_gp(self, mu, b_diag, X):
        """KL( N(μ, XXᵀ) || N(0, K) ) per batch item and latent dim, averaged per element."""
        B, L, D = mu.shape
        K_inv, logdet_K = self._prior(L, mu.device)
        tr      = torch.einsum('lm,bdmn,bdln->bd', K_inv, X, X)          # tr(K⁻¹ X Xᵀ)
        quad    = torch.einsum('bld,lm,bmd->bd', mu, K_inv, mu)
        logdetS = -2 * torch.log(b_diag).sum(dim=1)                      # log|Λ⁻¹| = −2 Σ log B_ll
        kl = 0.5 * (tr + quad - L + logdet_K - logdetS)
        return kl.sum() / (B * L * D)

    def loss_parts(self, x, mask):
        mu, b_diag, b_off = self.encode(x, mask)
        X = self.binv(b_diag, b_off)
        x_hat = self.dec(self.sample_z(mu, X))
        tgt = (1 - mask).bool()
        rec = F.mse_loss(x_hat[tgt], x[tgt]) if tgt.any() else torch.tensor(0., device=x.device)
        return rec, self.kl_gp(mu, b_diag, X)

    def loss(self, x, mask, kl_scale=1.0):
        rec, kl = self.loss_parts(x, mask)
        return rec + kl_scale * self.kl_w * kl


def train_gpvae_banded(df, path=GPVAE_MODEL, kl_w=GPVAE_KL_WEIGHT, seed=SPLIT_SEED):
    """75 epochs, shared LR schedule; KL weight ramped linearly from 0 over KL_WARMUP_EPOCHS."""
    if os.path.exists(path):
        model = GPVAEBanded(kl_weight=kl_w).to(DEVICE)
        model.load_state_dict(torch.load(path, map_location=DEVICE)); model.eval()
        print(f"  GP-VAE (banded posterior) loaded: {os.path.basename(path)} | KL weight {kl_w:g}")
        return model
    np.random.seed(seed); torch.manual_seed(seed)
    model = GPVAEBanded(kl_weight=kl_w).to(DEVICE)
    ld  = DataLoader(CGMDatasetSeq(df), batch_size=BATCH_SIZE, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = lr_schedule(opt)
    for ep in range(EPOCHS):
        model.train()
        kl_scale = min(1.0, (ep + 1) / KL_WARMUP_EPOCHS)          # KL warm-up
        er = ek = 0.0
        for x, m in ld:
            x, m = x.to(DEVICE), m.to(DEVICE)
            opt.zero_grad()
            rec, kl = model.loss_parts(x, m)
            (rec + kl_scale * kl_w * kl).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.); opt.step()
            er += rec.item(); ek += kl.item()
        sched(ep)
        if log_epoch(ep):
            print(f"    GP-VAE Ep{ep+1}/{EPOCHS} | rec {er/len(ld):.4f} | KL/elem {ek/len(ld):.2f}"
                  f" | KL scale {kl_scale:.1f}", flush=True)
    model.eval()
    torch.save(model.state_dict(), path)
    print(f"  saved: {path}")
    return model


def impute_gpvae_banded(masked_df, gap_info, model):
    """Same window, normalisation and output format as impute_gpvae; posterior mean for the
    point estimate and N_SAMPLES draws from the banded posterior for CRPS."""
    results = {}; model.eval()
    for g in gap_info:
        pid = g['pid']; gl = g['gap_len']
        pat = masked_df[masked_df['PtID'] == pid].sort_values('ts').reset_index(drop=True)
        idx = g['start_idx']; ws = max(0, idx - CTX_WIN//5); we = min(len(pat), idx + gl + CTX_WIN//5)
        seg = pat.iloc[ws:we]; L = len(seg)
        if L > MAX_L:
            results[id(g)] = {'glucose': [np.nan]*gl, 'hr': [np.nan]*gl}; continue
        gn = (seg['GlucoseCGM'].values - G_MEAN) / G_STD
        hn = (seg['HR'].values - H_MEAN) / H_STD
        sig = np.stack([gn, hn], axis=1)
        msk = (~np.isnan(sig)).astype(np.float32)
        sig = np.nan_to_num(sig, nan=0.)
        x = torch.tensor(sig, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        m = torch.tensor(msk, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.inference_mode():
            mu, b_diag, b_off = model.encode(x, m)
            out = model.dec(mu).squeeze(0).cpu().numpy()                     # (L, NF)
            X = model.binv(b_diag, b_off)
            gp_sn = model.dec(model.sample_z(mu, X, n=N_SAMPLES)).cpu().numpy()  # (N, L, NF)
        li = idx - ws
        results[id(g)] = {
            'glucose':         (out[li:li+gl, 0]*G_STD + G_MEAN).tolist(),
            'hr':              (out[li:li+gl, 1]*H_STD + H_MEAN).tolist(),
            'glucose_samples': (gp_sn[:, li:li+gl, 0]*G_STD + G_MEAN).astype(np.float32),
            'hr_samples':      (gp_sn[:, li:li+gl, 1]*H_STD + H_MEAN).astype(np.float32)}
    return results

print("GP-VAE ready.")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 8 — CATSI (Yin, Feng & Cheung, J. Healthc. Inform. Res. 2020), using the
# OFFICIAL model code:
#   https://github.com/ai4h-hkbu/CATSI  (model.py, utils.py, main.py; commit 239df11)
#
# Identical to the official code:
#   • the model (MLPFeatureImputation, InputTemporalDecay, RNNContext, CATSI),
#     hidden_size = 64, context_hidden = 32 (official defaults);
#   • the input construction of utils.py: missing values filled with the series
#     mean, observation-gap matrix δ (construct_delta_matrix), per-series min–max
#     from the ground truth in training and from the observed values at test time;
#   • the training recipe of main.py: loss on OBSERVED values only (the masked
#     block is held out, as DACMI's evaluation entries), Adam with lr 1e-3 and
#     weight decay 5e-3 (context GRU) / 5e-5 (rest), batch size 64, 75 epochs
#     (the official default is 300; set CATSI_EPOCHS = 300 to use it),
#     no LR schedule, 80/20 train/validation split of the training series, and
#     clip_grad_norm_ called before backward() exactly as in main.py.
#   • the final imputation = model output 'imputations' (deterministic → CRPS = MAE).
#
# Adapted to this benchmark (necessary, not a change to CATSI):
#   • a "series" is one window: training series are the same 48-step windows
#     (one contiguous 35-min block held out) used by every learned method, and each
#     test gap is imputed from the same ±120-min window as all other methods;
#   • time stamps in hours (5 min = 1/12 h) for δ;
#   • GUARDS for degenerate windows only (a variable with < 2 observed values or
#     max = min), marked "GUARD" below; they do not change any other computation.
#
# ONE CORRECTION to the released code (marked "FIX"): the series mean x̄ is computed
# as the mean of the OBSERVED values, as defined in the paper (Eq. 5, "the empirical
# mean"). The released code divides the sum of ALL values (observed + mean-filled)
# by the number of observed values, i.e. x̄ · L / n_observed. With DACMI's ~12%
# missingness this inflates x̄ by ~14%; in our windows HR is often almost entirely
# missing (e.g. 2 observed values in 144 steps → x̄ inflated 72×), the normalised
# mean then lies far outside [0, 1], and CATSI's cross-feature imputation of glucose
# from HR explodes (glucose RMSE > 2,000 mg/dL with the released formula). On synthetic data with
# 2 observed HR values per window: released code ≈ 900 mg/dL, corrected ≈ 14 mg/dL.
# ════════════════════════════════════════════════════════════════════
from torch.autograd import Variable
from torch.nn.parameter import Parameter
from torch.nn.utils import clip_grad_norm_
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim

CATSI_EPOCHS     = EPOCHS   # 75, same budget as every other learned model (official default: 300)
CATSI_BATCH      = 64       # main.py default
CATSI_VALID_FRAC = 0.2      # main.py / utils.load_data default (validation is only monitored)
CATSI_STEP_HOURS = 5.0 / 60.0


# ─────────────── official model.py (verbatim except the marked GUARD lines) ───────────────
class MLPFeatureImputation(nn.Module):
    def __init__(self, input_size, hidden_size=32):
        super(MLPFeatureImputation, self).__init__()

        self.W = Parameter(torch.Tensor(input_size, hidden_size, input_size))
        self.b = Parameter(torch.Tensor(input_size, hidden_size))
        self.nonlinear_regression = nn.Sequential(
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )

        m = torch.ones(input_size, hidden_size, input_size)
        stdv = 1. / math.sqrt(input_size)
        for i in range(input_size):
            m[i, :, i] = 0
        self.register_buffer('m', m)
        self.W.data.uniform_(-stdv, stdv)
        if self.b is not None:
            self.b.data.uniform_(-stdv, stdv)

    def forward(self, x):
        hidden = torch.cat(tuple(F.linear(x, self.W[i] * Variable(self.m[i]), self.b[i]).unsqueeze(2)
                            for i in range(len(self.W))), dim=2)
        z_h = self.nonlinear_regression(hidden)
        return z_h.squeeze(-1)


class InputTemporalDecay(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.W = Parameter(torch.Tensor(input_size, input_size))
        self.b = Parameter(torch.Tensor(input_size))

        m = torch.eye(input_size, input_size)
        self.register_buffer('m', m)

        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1. / math.sqrt(self.W.size(0))
        self.W.data.uniform_(-stdv, stdv)
        if self.b is not None:
            self.b.data.uniform_(-stdv, stdv)

    def forward(self, d):
        gamma = F.relu(F.linear(d, self.W * Variable(self.m), self.b))
        return torch.exp(-gamma)


class RNNContext(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.rnn_cell = nn.GRUCell(input_size, hidden_size)

    def forward(self, input, seq_lengths):
        T_max = input.shape[1]  # batch x time x dims

        h = torch.zeros(input.shape[0], self.hidden_size).to(input.device)
        hn = torch.zeros(input.shape[0], self.hidden_size).to(input.device)

        for t in range(T_max):
            h = self.rnn_cell(input[:, t, :], h)
            padding_mask = ((t + 1) <= seq_lengths).float().unsqueeze(1).to(input.device)
            hn = padding_mask * h + (1-padding_mask) * hn

        return hn


class CATSI(nn.Module):
    def __init__(self, num_vars, hidden_size=64, context_hidden=32):
        super().__init__()
        self.num_vars = num_vars
        self.hidden_size = hidden_size

        self.context_mlp = nn.Sequential(
            nn.Linear(3*self.num_vars+1, 2*context_hidden),
            nn.ReLU(),
            nn.Linear(2*context_hidden, context_hidden)
        )
        self.context_rnn = RNNContext(2*self.num_vars, context_hidden)

        self.initial_hidden = nn.Linear(2*context_hidden, 2*hidden_size)
        self.initial_cell_state = nn.Tanh()

        self.rnn_cell_forward = nn.LSTMCell(2*num_vars+2*context_hidden, hidden_size)
        self.rnn_cell_backward = nn.LSTMCell(2*num_vars+2*context_hidden, hidden_size)

        self.decay_inputs = InputTemporalDecay(input_size=num_vars)

        self.recurrent_impute = nn.Linear(2*hidden_size, num_vars)
        self.feature_impute = MLPFeatureImputation(num_vars)

        self.fuse_imputations = nn.Linear(2*num_vars, num_vars)

    def forward(self, data):
        seq_lengths = data['lengths']

        values = data['values']  # pts x time_stamps x vars
        masks = data['masks']
        deltas = data['deltas']

        # compute context vector, h0 and c0
        T_max = values.shape[1]
        padding_masks = torch.cat(tuple(((t + 1) <= seq_lengths).float().unsqueeze(1).to(values.device)
                                   for t in range(T_max)), dim=1)
        padding_masks = padding_masks.unsqueeze(2).repeat(1, 1, values.shape[2])  # pts x time_stamps x vars

        n_obs = masks.sum(dim=1).clamp(min=1)                                    # GUARD: no observed value
        data_means = (values * masks).sum(dim=1) / n_obs  # FIX: mean of observed values (paper Eq. 5)
        data_variance = ((values - data_means.unsqueeze(1)) ** 2).sum(dim=1) / (n_obs - 1).clamp(min=1)  # GUARD: < 2 observed
        data_stdev = data_variance ** 0.5
        data_missing_rate = 1 - masks.sum(dim=1) / padding_masks.sum(dim=1)
        data_stats = torch.cat((seq_lengths.unsqueeze(1).float(), data_means, data_stdev, data_missing_rate), dim=1)

        # normalization
        min_max_norm = data['max_vals'] - data['min_vals']
        min_max_norm = torch.where(min_max_norm > 0, min_max_norm, torch.ones_like(min_max_norm))  # GUARD: max = min
        normalized_values = (values - data['min_vals']) / min_max_norm
        normalized_means = (data_means - data['min_vals'].squeeze(1)) / min_max_norm.squeeze(1)

        if self.training:
            normalized_evals = (data['evals'] - data['min_vals']) / min_max_norm

        x_prime = torch.zeros_like(normalized_values)
        x_prime[:, 0, :] = normalized_values[:, 0, :]
        for t in range(1, T_max):
            x_prime[:, t, :] = normalized_values[:, t-1, :]

        gamma = self.decay_inputs(deltas)
        x_decay = gamma * x_prime + (1 - gamma) * normalized_means.unsqueeze(1)
        x_complement = (masks * normalized_values + (1-masks) * x_decay) * padding_masks

        context_mlp = self.context_mlp(data_stats)
        context_rnn = self.context_rnn(torch.cat((x_complement, deltas), dim=-1), seq_lengths)
        context_vec = torch.cat((context_mlp, context_rnn), dim=1)
        h = self.initial_hidden(context_vec)
        c = self.initial_cell_state(h)

        inputs = torch.cat([x_complement, masks, context_vec.unsqueeze(1).repeat(1, T_max, 1)], dim=-1)

        h_forward, c_forward = h[:, :self.hidden_size], c[:, :self.hidden_size]
        h_backward, c_backward = h[:, self.hidden_size:], c[:, self.hidden_size:]
        hiddens_forward = h[:, :self.hidden_size].unsqueeze(1)
        hiddens_backward = h[:, self.hidden_size:].unsqueeze(1)
        for t in range(T_max-1):
            h_forward, c_forward = self.rnn_cell_forward(inputs[:, t, :],
                                                         (h_forward, c_forward))
            h_backward, c_backward = self.rnn_cell_backward(inputs[:, T_max-1-t, :],
                                                            (h_backward, c_backward))
            hiddens_forward = torch.cat((hiddens_forward, h_forward.unsqueeze(1)), dim=1)
            hiddens_backward = torch.cat((h_backward.unsqueeze(1), hiddens_backward), dim=1)

        rnn_imp = self.recurrent_impute(torch.cat((hiddens_forward, hiddens_backward), dim=2))
        feat_imp = self.feature_impute(x_complement).squeeze(-1)

        # imputation fusion
        beta = torch.sigmoid(self.fuse_imputations(torch.cat((gamma, masks), dim=-1)))
        imp_fusion = beta * feat_imp + (1 - beta) * rnn_imp
        final_imp = masks * normalized_values + (1-masks) * imp_fusion

        rnn_loss = F.mse_loss(rnn_imp * masks, normalized_values * masks, reduction='sum')
        feat_loss = F.mse_loss(feat_imp * masks, normalized_values * masks, reduction='sum')
        fusion_loss = F.mse_loss(imp_fusion * masks, normalized_values * masks, reduction='sum')
        total_loss = rnn_loss + feat_loss + fusion_loss

        if self.training:
            rnn_loss_eval = F.mse_loss(rnn_imp * data['eval_masks'], normalized_evals * data['eval_masks'], reduction='sum')
            feat_loss_eval = F.mse_loss(feat_imp * data['eval_masks'], normalized_evals * data['eval_masks'], reduction='sum')
            fusion_loss_eval = F.mse_loss(imp_fusion * data['eval_masks'], normalized_evals * data['eval_masks'], reduction='sum')
            total_loss_eval = rnn_loss_eval + feat_loss_eval + fusion_loss_eval

        def rescale(x):
            return torch.where(padding_masks==1, x * min_max_norm + data['min_vals'], padding_masks)

        feat_imp = rescale(feat_imp)
        rnn_imp = rescale(rnn_imp)
        final_imp = rescale(final_imp)

        out_dict = {
            'loss': total_loss / masks.sum(),
            'verbose_loss': [
                ('rnn_loss', rnn_loss / masks.sum(), masks.sum()),
                ('feat_loss', feat_loss / masks.sum(), masks.sum()),
                ('fusion_loss', fusion_loss / masks.sum(), masks.sum())
            ],
            'loss_count': masks.sum(),
            'imputations': final_imp,
            'feat_imp': feat_imp,
            'hist_imp': rnn_imp
        }
        if self.training:
            out_dict['loss_eval'] = total_loss_eval / data['eval_masks'].sum()
            out_dict['loss_eval_count'] = data['eval_masks'].sum()
            out_dict['verbose_loss'] += [
                ('rnn_loss_eval', rnn_loss_eval / data['eval_masks'].sum(), data['eval_masks'].sum()),
                ('feat_loss_eval', feat_loss_eval / data['eval_masks'].sum(), data['eval_masks'].sum()),
                ('fusion_loss_eval', fusion_loss_eval / data['eval_masks'].sum(), data['eval_masks'].sum())
            ]

        return out_dict


# ─────────────── official utils.py data construction, applied to one window ───────────────
_CATSI_TRAIN_MIN = np.array([train_df['GlucoseCGM'].min(), train_df['HR'].min()], float)
_CATSI_TRAIN_MAX = np.array([train_df['GlucoseCGM'].max(), train_df['HR'].max()], float)
_CATSI_TRAIN_MEAN = np.array([G_MEAN, H_MEAN], float)

def _catsi_fill_col_mean(x):
    """utils.replace_nan_with_col_mean; GUARD: a column with no observed value uses the TRAIN mean."""
    x = x.copy()
    missing_flag = np.isnan(x)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        col_means = np.nanmean(x, axis=0)
    col_means = np.where(np.isfinite(col_means), col_means, _CATSI_TRAIN_MEAN)
    idx = np.where(missing_flag)
    x[idx] = np.take(col_means, idx[1])
    return x, missing_flag

def _catsi_minmax(x):
    """np.nanmax / np.nanmin per column; GUARD: all-missing column uses the TRAIN range."""
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        mx, mn = np.nanmax(x, axis=0), np.nanmin(x, axis=0)
    mx = np.where(np.isfinite(mx), mx, _CATSI_TRAIN_MAX)
    mn = np.where(np.isfinite(mn), mn, _CATSI_TRAIN_MIN)
    return mx.reshape(1, -1), mn.reshape(1, -1)

def _catsi_delta(time_stamps, observed_mask):
    """utils.construct_delta_matrix."""
    delta = np.zeros(observed_mask.shape)
    for t in range(1, observed_mask.shape[0]):
        delta[t, :] = time_stamps[t] - time_stamps[t - 1] + (1 - observed_mask[t, :]) * delta[t - 1, :]
    return delta

def _catsi_item(x_input, x_truth=None, pid=0):
    """One series in the format of utils.load_raw_data. x_input: (L, 2) raw values with NaN
    where missing; x_truth: (L, 2) ground truth for training (load_ith_pt_training), or None
    at test time (load_ith_pt_testing)."""
    L = x_input.shape[0]
    time_stamps = np.arange(L) * CATSI_STEP_HOURS
    pt_with_na, missing_flag = _catsi_fill_col_mean(x_input)
    observed_mask = (~missing_flag).astype(float)
    item = {'pt_with_na': pt_with_na, 'time_stamps': time_stamps, 'observed_mask': observed_mask,
            'length': L, 'delta': _catsi_delta(time_stamps, observed_mask), 'pid': pid}
    if x_truth is not None:                                    # training: range from the ground truth
        item['pt_max'], item['pt_min'] = _catsi_minmax(x_truth)
        gt, missing_flag_gt = _catsi_fill_col_mean(x_truth)
        item['pt_ground_truth'] = gt
        item['eval_mask'] = ((~missing_flag_gt) & missing_flag).astype(float)
    else:                                                      # testing: range from observed values
        item['pt_max'], item['pt_min'] = _catsi_minmax(x_input)
    return item

def _catsi_collate(batch_data, testing=False, device=DEVICE):
    """utils.build_data_loader.pad_time_series_batch."""
    lengths = [x['length'] for x in batch_data]
    pids = [x['pid'] for x in batch_data]
    lengths, data_idx = torch.sort(torch.LongTensor(lengths), descending=True)
    batch_data = [batch_data[idx] for idx in data_idx]
    pids = [pids[idx] for idx in data_idx]
    d = {}
    d['values'] = pad_sequence([torch.FloatTensor(x['pt_with_na']) for x in batch_data], batch_first=True).to(device)
    d['masks'] = pad_sequence([torch.FloatTensor(x['observed_mask']) for x in batch_data], batch_first=True).to(device)
    d['deltas'] = pad_sequence([torch.FloatTensor(x['delta']) for x in batch_data], batch_first=True).to(device)
    d['time_stamps'] = pad_sequence([torch.FloatTensor(x['time_stamps']) for x in batch_data], batch_first=True).to(device)
    d['lengths'] = lengths.to(device)
    d['pids'] = pids
    d['max_vals'] = torch.FloatTensor(np.concatenate([x['pt_max'] for x in batch_data])).to(device).unsqueeze(1)
    d['min_vals'] = torch.FloatTensor(np.concatenate([x['pt_min'] for x in batch_data])).to(device).unsqueeze(1)
    if not testing:
        d['evals'] = pad_sequence([torch.FloatTensor(x['pt_ground_truth']) for x in batch_data], batch_first=True).to(device)
        d['eval_masks'] = pad_sequence([torch.FloatTensor(x['eval_mask']) for x in batch_data], batch_first=True).to(device)
    return d


def _catsi_training_series(df, seed=SPLIT_SEED):
    """The same 48-step training windows as every learned method (CGMDatasetSeq), with one
    contiguous 35-min block held out per window (fixed once, like DACMI's evaluation entries)."""
    ds = CGMDatasetSeq(df)
    torch.manual_seed(seed)
    items = []
    for i in range(len(ds)):
        x, m = ds[i]
        truth = x.numpy() * np.array([G_STD, H_STD]) + np.array([G_MEAN, H_MEAN])   # raw mg/dL, bpm
        inp = truth.copy(); inp[m.numpy() == 0] = np.nan
        items.append(_catsi_item(inp, truth, pid=i))
    return items


@torch.no_grad()
def _catsi_eval_masked(model, items):
    """MAE (raw units) on the held-out block, per signal (as main.evaluate, MAE part)."""
    model.eval(); err = np.zeros(2); n = np.zeros(2)
    for b in range(0, len(items), 256):
        d = _catsi_collate(items[b:b+256])
        imp = model(d)['imputations']
        em = d['eval_masks']
        err += (em * (d['evals'] - imp).abs()).sum(dim=[0, 1]).cpu().numpy()
        n += em.sum(dim=[0, 1]).cpu().numpy()
    return err / np.maximum(n, 1)


def train_catsi_official(df, path=CATSI_MODEL):
    model = CATSI(NF).to(DEVICE)
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=DEVICE)); model.eval()
        print(f"  CATSI (official) loaded: {os.path.basename(path)}")
        return model
    np.random.seed(SPLIT_SEED); torch.manual_seed(SPLIT_SEED); random.seed(SPLIT_SEED)
    items = _catsi_training_series(df)
    random.shuffle(items)                                            # utils.load_data
    n_valid = int(CATSI_VALID_FRAC * len(items))
    train_items, valid_items = items[:len(items) - n_valid], items[len(items) - n_valid:]
    print(f"  CATSI: {len(train_items)} training / {len(valid_items)} validation series "
          f"(48 steps each), {CATSI_EPOCHS} epochs, batch {CATSI_BATCH}")

    optimizer = optim.Adam([                                         # main.fit
        {'params': model.context_rnn.parameters(), 'lr': 1e-3, 'weight_decay': 5e-3},
        {'params': [p for n, p in model.named_parameters() if n.split('.')[0] != 'context_rnn'],
         'lr': 1e-3, 'weight_decay': 5e-5}])

    for epoch in range(CATSI_EPOCHS):
        model.train()
        order = list(range(len(train_items))); random.shuffle(order)
        tl = tc = el = ec = 0.0
        for b in range(0, len(order), CATSI_BATCH):
            data = _catsi_collate([train_items[i] for i in order[b:b+CATSI_BATCH]])
            optimizer.zero_grad()
            ret = model(data)
            clip_grad_norm_(model.parameters(), 1)                   # as in main.py (before backward)
            ret['loss'].backward()
            optimizer.step()
            tl += ret['loss'].item() * ret['loss_count'].item(); tc += ret['loss_count'].item()
            el += ret['loss_eval'].item() * ret['loss_eval_count'].item(); ec += ret['loss_eval_count'].item()
        if epoch == 0 or (epoch + 1) % 10 == 0 or (epoch + 1) == CATSI_EPOCHS:
            msg = f"    CATSI Ep{epoch+1}/{CATSI_EPOCHS} | train loss {tl/tc:.4e} | held-out block loss {el/ec:.4e}"
            if valid_items:
                mae = _catsi_eval_masked(model, valid_items)
                msg += f" | validation block MAE glucose {mae[0]:.2f} mg/dL, HR {mae[1]:.2f} bpm"
            print(msg, flush=True)
    model.eval()
    torch.save(model.state_dict(), path)
    print(f"  saved: {path}")
    return model


def impute_catsi(masked_df, gap_info, model):
    """Same ±120-min window as all other methods; test-time series construction of utils.py."""
    results = {}; model.eval()
    for g in gap_info:
        pid = g['pid']; gl = g['gap_len']
        pat = masked_df[masked_df['PtID'] == pid].sort_values('ts').reset_index(drop=True)
        idx = g['start_idx']; ws = max(0, idx - CTX_WIN//5); we = min(len(pat), idx + gl + CTX_WIN//5)
        seg = pat.iloc[ws:we]; L = len(seg)
        if L > MAX_L:
            results[id(g)] = {'glucose': [np.nan]*gl, 'hr': [np.nan]*gl}; continue
        raw = seg[['GlucoseCGM', 'HR']].to_numpy(float)
        with torch.no_grad():
            out = model(_catsi_collate([_catsi_item(raw)], testing=True))['imputations'][0].cpu().numpy()
        li = idx - ws
        results[id(g)] = {'glucose': out[li:li+gl, 0].tolist(), 'hr': out[li:li+gl, 1].tolist()}
    return results

print("CATSI ready.")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 9 — CSDI-adapted (conditional score-based diffusion, Tashiro et al., NeurIPS 2021)
#   Input (B, 3·NF, L) = [noisy targets | clean observed values | mask]; residual blocks
#   with temporal attention and a feature MLP; cosine noise schedule, T = 100.
#   Trained on the same 48-step windows with one contiguous masked block, loss on the
#   masked positions; 75 epochs with the shared LR schedule. Inference: noise
#   initialisation, full T = 100 reverse steps, 50 chains; median-of-chains and
#   best-chain point estimates.
# ════════════════════════════════════════════════════════════════════
D_TIME_EMB    = 128
CSDI_D_MODEL  = 64
CSDI_N_HEADS  = 4
CSDI_N_LAYERS = 4


class DiffusionEmbedding(nn.Module):
    def __init__(self, d_time=D_TIME_EMB, d_out=CSDI_D_MODEL):
        super().__init__()
        self.d_time = d_time
        self.proj1  = nn.Linear(d_time, d_out)
        self.proj2  = nn.Linear(d_out,  d_out)
    def forward(self, t):
        half = self.d_time // 2
        freq = torch.exp(-np.log(10000)*torch.arange(half, device=t.device, dtype=torch.float)/half)
        args = t[:, None].float() * freq[None]
        emb  = torch.cat([args.sin(), args.cos()], dim=-1)
        return torch.nn.functional.silu(self.proj2(torch.nn.functional.silu(self.proj1(emb))))


class CSDIResBlock(nn.Module):
    def __init__(self, d_model=CSDI_D_MODEL, n_heads=CSDI_N_HEADS, nf=NF):
        super().__init__()
        self.t_proj    = nn.Linear(d_model, d_model)
        self.conv1     = nn.Conv1d(d_model, d_model*2, kernel_size=3, padding=1)
        self.temp_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.temp_ln   = nn.LayerNorm(d_model)
        self.feat_fc   = nn.Sequential(
            nn.Linear(d_model, d_model*2), nn.GELU(), nn.Linear(d_model*2, d_model))
        self.feat_ln   = nn.LayerNorm(d_model)
        self.out_proj  = nn.Conv1d(d_model, d_model, kernel_size=1)
        self.skip_proj = nn.Conv1d(d_model, d_model, kernel_size=1)
    def forward(self, x, t_emb):
        B, D, L = x.shape
        h = x + self.t_proj(t_emb).unsqueeze(-1)
        h = self.conv1(h); h1, h2 = h.chunk(2, dim=1)
        h = torch.tanh(h1) * torch.sigmoid(h2)
        ht = h.permute(0,2,1); ht_attn,_ = self.temp_attn(ht,ht,ht)
        ht = self.temp_ln(ht + ht_attn)
        hf = self.feat_ln(ht + self.feat_fc(ht)).permute(0,2,1)
        return (self.out_proj(hf) + x) / np.sqrt(2), self.skip_proj(hf)


class CSDI(nn.Module):
    """CSDI-adapted conditional diffusion baseline used in this study."""
    def __init__(self, nf=NF, d_model=CSDI_D_MODEL, n_heads=CSDI_N_HEADS,
                 n_layers=CSDI_N_LAYERS, T=T_STEPS):
        super().__init__()
        self.T = T; self.n_layers = n_layers
        self.input_proj  = nn.Conv1d(nf*3, d_model, kernel_size=1)
        self.t_emb       = DiffusionEmbedding(D_TIME_EMB, d_model)
        self.layers      = nn.ModuleList([CSDIResBlock(d_model, n_heads, nf)
                                          for _ in range(n_layers)])
        self.output_proj = nn.Sequential(
            nn.Conv1d(d_model, d_model, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(d_model, nf, kernel_size=1))
        betas = self._cs(T)
        self.register_buffer('alpha_bar', torch.cumprod(1-betas, 0))
    def _cs(self, T, s=0.008):
        st = torch.arange(T+1, dtype=torch.float)
        f  = torch.cos(((st/T+s)/(1+s))*np.pi/2)**2; f = f/f[0]
        return torch.clamp(1-f[1:]/f[:-1], 0.0001, 0.9999)
    def forward(self, x_noisy, t, x_observed, mask):
        inp = torch.cat([x_noisy, x_observed, mask], dim=1)
        h   = self.input_proj(inp); t_emb = self.t_emb(t)
        skip_sum = torch.zeros_like(h)
        for layer in self.layers:
            h, skip = layer(h, t_emb); skip_sum += skip
        return self.output_proj(skip_sum / np.sqrt(self.n_layers))
    def loss(self, x0, mask):
        # Corrected (Tashiro et al.): x0 = FULL clean signal.
        # Diffuse full x0; x_noisy channel carries noisy TARGETS only;
        # x_observed channel carries clean observed values;
        # loss at TARGET positions.
        B = x0.shape[0]; t = torch.randint(1, self.T+1, (B,), device=x0.device)
        ab = self.alpha_bar[t-1].view(B,1,1); eps = torch.randn_like(x0)
        x_n = ab.sqrt()*x0 + (1-ab).sqrt()*eps
        pred = self.forward(x_n*(1-mask), t, x0*mask, mask)
        miss = 1 - mask
        if miss.sum() > 0:
            return torch.nn.functional.mse_loss(pred*miss, eps*miss)
        return torch.nn.functional.mse_loss(pred*mask, eps*mask)


class CSDIDataset(Dataset):
    def __init__(self, df, seq=SEQ_LEN, mr=MASK_RATIO):
        self.s = []
        for pid,p in df.groupby('PtID'):
            p = p.sort_values('ts').reset_index(drop=True)
            g = (p['GlucoseCGM'].values-G_MEAN)/G_STD
            h = (p['HR'].values-H_MEAN)/H_STD
            sig = np.stack([g,h], axis=0)   # (NF, T)
            for i in range(0, sig.shape[1]-seq, seq//2):
                w = sig[:, i:i+seq]
                if np.isnan(w).any(): continue
                self.s.append(w)
        self.s = limit_windows(self.s)
    def __len__(self): return len(self.s)
    def __getitem__(self, i):
        x = torch.tensor(self.s[i], dtype=torch.float32)  # (NF, seq) FULL clean
        m = torch.ones_like(x)
        # Contiguous block masking, as for every other learned method.
        nm = max(1, int(SEQ_LEN*MASK_RATIO))
        st = int(torch.randint(0, SEQ_LEN - nm + 1, (1,)).item())
        m[:, st:st+nm] = 0
        return x, m   # FULL clean signal — loss masks targets internally


class CSDIBndDataset(CSDIDataset):
    """Ablation only (csdi_bnd): CAKGI's masking regime (50 % anchored / 50 % free block) and
    CAKGI's boundary weights, so CSDI-adapted can be trained with the boundary-aware loss."""
    def __getitem__(self, i):
        x = torch.tensor(self.s[i], dtype=torch.float32); m = torch.ones_like(x)
        nm = max(1, int(SEQ_LEN*MASK_RATIO))
        if np.random.random() < 0.5:
            st = int(np.random.randint(1, max(1, SEQ_LEN - nm - 1) + 1))
        else:
            st = int(np.random.randint(0, SEQ_LEN - nm + 1))
        en = st + nm - 1; m[:, st:en+1] = 0
        bnd = torch.zeros(SEQ_LEN)
        for b in range(st, en + 1):
            d = min(b - st, en - b)
            if d < 3: bnd[b] = 1.0 - d/3.0
        return x, m, bnd


def train_csdi(df, path=CSDI_MODEL, boundary_loss=False, tag="CSDI-adapted"):
    model = CSDI().to(DEVICE)
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=DEVICE)); model.eval()
        print(f"  {tag} loaded: {os.path.basename(path)}")
        return model
    np.random.seed(SPLIT_SEED); torch.manual_seed(SPLIT_SEED)
    ds  = CSDIBndDataset(df) if boundary_loss else CSDIDataset(df)
    ld  = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = lr_schedule(opt)
    model.train()
    for ep in range(EPOCHS):
        tot = 0.0
        for batch in ld:
            opt.zero_grad()
            if boundary_loss:                                   # + CAKGI boundary-aware loss (0.3)
                x, m, bnd = (b.to(DEVICE) for b in batch)
                l = model.loss(x, m)
                B = x.shape[0]
                t = torch.randint(1, model.T + 1, (B,), device=DEVICE)
                a = model.alpha_bar[t-1].view(B, 1, 1); eps = torch.randn_like(x)
                pred = model((a.sqrt()*x + (1-a).sqrt()*eps)*(1-m), t, x*m, m)
                w = bnd.view(B, 1, -1).expand_as(eps)*(1-m)
                l = l + 0.3 * (((pred - eps)**2)*w).sum()/w.sum().clamp(min=1)
            else:                                               # standard CSDI loss
                x, m = (b.to(DEVICE) for b in batch)
                l = model.loss(x, m)
            l.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += l.item()
        sched(ep)
        if log_epoch(ep):
            print(f"    {tag} Ep{ep+1}/{EPOCHS} | {tot/len(ld):.4f}", flush=True)
    model.eval()
    torch.save(model.state_dict(), path)
    print(f"  saved: {path}")
    return model


def sample_csdi(model, xi, mt, n=N_SAMPLES):
    """Batched CSDI sampling. xi, mt: (1, NF, L). Returns (N,1,NF,L) on CPU."""
    model.eval()
    out_chunks = []
    with torch.inference_mode():
        for nbatch in chain_chunks(n, CHAIN_BATCH_SIZE):
            xi_b = xi.expand(nbatch,-1,-1).contiguous()
            mt_b = mt.expand(nbatch,-1,-1).contiguous()
            x = torch.randn_like(xi_b)
            x = x*(1-mt_b) + xi_b*mt_b
            for t in reversed(range(1, model.T+1)):
                tt  = torch.full((nbatch,), t, device=DEVICE, dtype=torch.long)
                eps = model(x*(1-mt_b), tt, xi_b*mt_b, mt_b)
                ab  = model.alpha_bar[t-1]
                abp = model.alpha_bar[t-2] if t>1 else torch.tensor(1., device=DEVICE)
                x0  = ((x-(1-ab).sqrt()*eps)/ab.sqrt()).clamp(-3,3)
                if t > 1:
                    x = abp.sqrt()*x0 + (1-abp).sqrt()*torch.randn_like(x)
                else:
                    x = x0
                x = x*(1-mt_b) + xi_b*mt_b
            out_chunks.append(x.unsqueeze(1).cpu())
    return torch.cat(out_chunks, dim=0)


def impute_csdi(masked_df, gap_info, model):
    results = {}; model.eval()
    for gap_i, g in enumerate(gap_info):
        pid=g['pid']; gl=g['gap_len']
        pat=masked_df[masked_df['PtID']==pid].sort_values('ts').reset_index(drop=True)
        idx=g['start_idx']; ws=max(0,idx-CTX_WIN//5); we=min(len(pat),idx+gl+CTX_WIN//5)
        seg=pat.iloc[ws:we]; L=len(seg)
        if L>MAX_L: results[id(g)]={'glucose':[np.nan]*gl,'hr':[np.nan]*gl}; continue
        gn=(seg['GlucoseCGM'].values-G_MEAN)/G_STD; hn=(seg['HR'].values-H_MEAN)/H_STD
        sig=np.stack([gn,hn],axis=0).astype(np.float32)   # (NF, L)
        msk=(~np.isnan(sig)).astype(np.float32)
        sig=np.nan_to_num(sig, nan=0.)
        xi=torch.tensor(sig, dtype=torch.float32).unsqueeze(0).to(DEVICE)   # (1,NF,L)
        mt=torch.tensor(msk, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        samps = sample_csdi(model, xi, mt, n=N_SAMPLES)
        sn    = samps[:,0,:,:].numpy()   # (N, NF, L)
        li    = idx - ws
        # Best chain — CSDI format is (N, NF, L) channels-first
        sn_seq = sn.transpose(0,2,1)            # (N, L, NF)
        xi_np  = xi.squeeze(0).cpu().numpy().T  # (L, NF)
        mt_np  = mt.squeeze(0).cpu().numpy().T  # (L, NF)
        best_i = select_best_chain(sn_seq, xi_np, mt_np, target_start=li, target_len=gl)
        best_s = sn[best_i]                     # (NF, L)
        med_s  = np.median(sn, axis=0)          # (NF, L) — official protocol
        results[id(g)] = {
            'glucose':          (best_s[0, li:li+gl]*G_STD+G_MEAN).tolist(),
            'hr':               (best_s[1, li:li+gl]*H_STD+H_MEAN).tolist(),
            'glucose_med':      (med_s[0, li:li+gl]*G_STD+G_MEAN).tolist(),
            'hr_med':           (med_s[1, li:li+gl]*H_STD+H_MEAN).tolist(),
            'glucose_samples':  (sn[:,0,li:li+gl]*G_STD+G_MEAN).astype(np.float32),
            'hr_samples':       (sn[:,1,li:li+gl]*H_STD+H_MEAN).astype(np.float32)}
        release_device_cache(gap_index=gap_i)
    return results

print("CSDI-adapted ready.")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 10 — CAKGI (PIDCK): context features, model, inference and training
#
#   Context (pre-gap vector, 133 dims): 12 immediate pre-gap statistics (incl. gap duration),
#   33 features of the same 2-h window 1, 3 and 7 days earlier, 21 contrasts current − history,
#   each with an observed-indicator, plus a constant. Post-gap vector (24 dims): 12 statistics
#   of the 2 h after the gap (incl. time of day and gap duration) with indicators.
#   Denoiser: 3 layers of self-attention + two cross-attentions (pre / post context) mixed by
#   learned gap-position weights λ_pre, λ_post (PosWeightNet); Cauchy-kernel regulariser on
#   the self-attention outputs. Inference: linear-interpolation initialisation, adaptive
#   T_eff = clip(2·gap_len, 20, 100), boundary guidance in the first half of the reverse
#   process, 50 chains; median-of-chains (primary) and best-chain point estimates.
# ════════════════════════════════════════════════════════════════════
PRE_COLS_ORIG = [
    'pre_glucose_mean','pre_glucose_min','pre_glucose_last',
    'pre_glucose_slope','pre_glucose_dropping',
    'pre_hypo_flag','pre_hypo_count','pre_time_since_hypo',
    'pre_hr_mean','pre_hr_last','pre_hr_slope','gap_duration_mins']
_HIST_F = ['glucose_mean','glucose_min','glucose_last','glucose_slope',
           'glucose_dropping','hypo_flag','hypo_count','time_since_hypo',
           'hr_mean','hr_last','hr_slope']
_CONT_F = ['glucose_mean','glucose_min','glucose_last','glucose_slope',
           'hr_mean','hr_last','hr_slope']
PRE_COLS_HIST     = [f'hist_{o}_{f}'     for o in ['1d','3d','7d'] for f in _HIST_F]
PRE_COLS_CONTRAST = [f'contrast_{o}_{f}' for o in ['1d','3d','7d'] for f in _CONT_F]
PRE_COLS  = PRE_COLS_ORIG + PRE_COLS_HIST + PRE_COLS_CONTRAST
POST_COLS = ['post_glucose_mean','post_glucose_min','post_glucose_first',
             'post_glucose_slope','post_hypo_flag','post_hypo_count',
             'post_hr_mean','post_hr_first','post_hr_slope','time_sin','time_cos',
             'gap_duration_mins']   # 12 features — gap duration added

HIST_OFFSETS_MC = {'1d':pd.Timedelta(days=1),'3d':pd.Timedelta(days=3),'7d':pd.Timedelta(days=7)}
HIST_TOL_MC     = pd.Timedelta(minutes=15)

def _ms(s):
    s=s.dropna(); return np.polyfit(np.arange(len(s)),s,1)[0] if len(s)>=2 else np.nan
def _tod(ts):
    h=ts.hour+ts.minute/60; return np.sin(2*np.pi*h/24),np.cos(2*np.pi*h/24)
def _exw(pat,cts,wm,pfx):
    w=pat[(pat['ts']>=cts-pd.Timedelta(minutes=wm))&(pat['ts']<cts)]
    pg=w['GlucoseCGM']; ph=w['HR']
    phf=int((pg<HYPO_THR).any()) if len(pg)>0 else 0
    phc=int((pg<HYPO_THR).sum()) if len(pg)>0 else 0
    phl=np.nan
    if phf:
        ht=w[w['GlucoseCGM']<HYPO_THR]['ts']; phl=(cts-ht.max()).total_seconds()/60
    return {f'{pfx}_glucose_mean':pg.mean(),f'{pfx}_glucose_min':pg.min() if len(pg)>0 else np.nan,
            f'{pfx}_glucose_last':pg.dropna().iloc[-1] if pg.dropna().size>0 else np.nan,
            f'{pfx}_glucose_slope':_ms(pg),f'{pfx}_glucose_dropping':int(_ms(pg)<-1.) if not np.isnan(_ms(pg)) else 0,
            f'{pfx}_hypo_flag':phf,f'{pfx}_hypo_count':phc,f'{pfx}_time_since_hypo':phl,
            f'{pfx}_hr_mean':ph.mean(),f'{pfx}_hr_last':ph.dropna().iloc[-1] if ph.dropna().size>0 else np.nan,
            f'{pfx}_hr_slope':_ms(ph)}
def _hc(pat, gs, cur):
    hf = {}; cf = {}
    core = ['glucose_mean','glucose_min','glucose_last',
            'glucose_slope','hr_mean','hr_last','hr_slope']
    for nm, dl in HIST_OFFSETS_MC.items():
        hts = gs - dl
        nr  = pat[(pat['ts'] >= hts - HIST_TOL_MC) &
                  (pat['ts'] <= hts + HIST_TOL_MC)]
        if len(nr) == 0:
            # No historical data at this anchor — fill with NaN
            # continue must be OUTSIDE the inner for loops
            for k in _HIST_F:
                hf[f'hist_{nm}_{k}'] = np.nan
            for k in core:
                cf[f'contrast_{nm}_{k}'] = np.nan
            continue   # outer loop continue — skip to next nm
        # Find closest timestamp to the historical anchor
        pos = np.argsort((nr['ts'] - hts).abs().values)
        if len(pos) == 0:
            for k in _HIST_F: hf[f'hist_{nm}_{k}'] = np.nan
            for k in core:    cf[f'contrast_{nm}_{k}'] = np.nan
            continue
        cl  = nr.iloc[pos[:1]]['ts'].values[0]
        hff = _exw(pat, pd.Timestamp(cl), CTX_WIN, f'hist_{nm}')
        hf.update(hff)
        for k in core:
            try:
                cf[f'contrast_{nm}_{k}'] = (float(cur.get(f'pre_{k}', np.nan)) -
                                             float(hff.get(f'hist_{nm}_{k}', np.nan)))
            except (TypeError, ValueError):
                cf[f'contrast_{nm}_{k}'] = np.nan
    return hf, cf


class SinPE(nn.Module):
    def __init__(self,d=64): super().__init__(); self.d=d
    def forward(self,t):
        h=self.d//2; f=torch.exp(-np.log(10000)*torch.arange(h,device=t.device)/h)
        a=t[:,None].float()*f[None]; return torch.cat([a.sin(),a.cos()],dim=-1)

class CtxEnc(nn.Module):
    def __init__(self,d_in,d=64):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(d_in,d),nn.LayerNorm(d),nn.GELU(),
                               nn.Linear(d,d),nn.LayerNorm(d),nn.GELU(),nn.Linear(d,d))
    def forward(self,x): return self.net(x).unsqueeze(1)


class PosWeightNet(nn.Module):
    def __init__(self,d=DM):
        super().__init__()
        self.p=nn.Sequential(nn.Linear(d*2+1,d//2),nn.GELU(),nn.Linear(d//2,1))
        self.q=nn.Sequential(nn.Linear(d*2+1,d//2),nn.GELU(),nn.Linear(d//2,1))
    def forward(self,x,cp,cq,gpos):
        B,L,D=x.shape
        fp=torch.cat([x,cp.expand(B,L,D),gpos],dim=-1)
        fq=torch.cat([x,cq.expand(B,L,D),gpos],dim=-1)
        w=F.softmax(torch.cat([self.p(fp)+(1-gpos),self.q(fq)+gpos],dim=-1),dim=-1)
        return w[...,0:1],w[...,1:2]

class DKLayer(nn.Module):
    def __init__(self,d=DM,h=NH):
        super().__init__()
        self.sa=nn.MultiheadAttention(d,h,batch_first=True)
        self.ca1=nn.MultiheadAttention(d,h,batch_first=True)
        self.ca2=nn.MultiheadAttention(d,h,batch_first=True)
        self.pw=PosWeightNet(d)
        self.ff=nn.Sequential(nn.Linear(d,d*4),nn.GELU(),nn.Linear(d*4,d))
        self.n1=nn.LayerNorm(d); self.n2=nn.LayerNorm(d); self.n3=nn.LayerNorm(d); self.sa_out=None
    def forward(self,x,cp,cq,gpos):
        x=self.n1(x+self.sa(x,x,x)[0]); self.sa_out=x
        lp,lq=self.pw(x,cp,cq,gpos); A1,_=self.ca1(x,cp,cp); A2,_=self.ca2(x,cq,cq)
        x=self.n2(x+lp*A1+lq*A2); x=self.n3(x+self.ff(x)); return x

class PIDCKModel(nn.Module):
    def __init__(self,PRE_DIM,POST_DIM):
        super().__init__(); self.T=T_STEPS
        self.ip=nn.Linear(NF*2,DM); self.te=SinPE(DM); self.tp=nn.Linear(DM,DM)
        self.pe=CtxEnc(PRE_DIM,DM); self.qe=CtxEnc(POST_DIM,DM)
        self.layers=nn.ModuleList([DKLayer() for _ in range(NL)]); self.op=nn.Linear(DM,NF)
        b=self._cs(T_STEPS); self.register_buffer('alpha_bar',torch.cumprod(1-b,0))
    def _cs(self,T,s=0.008):
        st=torch.arange(T+1,dtype=torch.float); f=torch.cos(((st/T+s)/(1+s))*np.pi/2)**2; f=f/f[0]
        return torch.clamp(1-f[1:]/f[:-1],0.0001,0.9999)
    def _gpos(self,mask,B,L):
        """Gap-position encoding; device-safe on CUDA, MPS and CPU."""
        obs=mask[:,:,0]
        gp=torch.full((B,L),0.5,device=mask.device,dtype=mask.dtype)
        for b in range(B):
            mis=torch.where(obs[b]==0)[0]
            if mis.numel()==0:
                continue

            # Breakpoints between missing runs; only the small index list goes to the CPU.
            brk=(torch.where(torch.diff(mis)>1)[0]+1).detach().cpu().tolist()
            starts=[0]+brk
            ends=brk+[int(mis.numel())]

            for s,e in zip(starts,ends):
                blk=mis[s:e]
                n=int(blk.numel())
                if n>1:
                    gp[b,blk]=torch.linspace(0.,1.,n,device=mask.device,dtype=mask.dtype)
                else:
                    gp[b,blk]=0.5
        return gp.unsqueeze(-1)
    def _cauchy(self,l=CAUCHY_L):
        """Cauchy temporal-kernel regularizer used by the original PIDCK training notebook."""
        tot=0.; cnt=0
        for layer in self.layers:
            z=layer.sa_out
            if z is None:
                continue
            B,L,D=z.shape
            zn=F.normalize(z,dim=-1)
            sim=torch.bmm(zn,zn.transpose(1,2))
            pos=torch.arange(L,dtype=torch.float,device=z.device)
            tgt=(1./(1.+(pos.unsqueeze(0)-pos.unsqueeze(1))**2/l**2)).unsqueeze(0).expand(B,-1,-1)
            tot += torch.nn.functional.mse_loss(sim,tgt)
            cnt += 1
        if cnt == 0:
            return torch.tensor(0.,device=next(self.parameters()).device)
        return tot/cnt

    def forward(self,xn,t,mask,cp_v,cq_v):
        B, L, NF_ = xn.shape
        x=self.ip(torch.cat([xn,mask],dim=-1))+self.tp(self.te(t)).unsqueeze(1)
        cp=self.pe(cp_v); cq=self.qe(cq_v); gp=self._gpos(mask,B,L)
        for layer in self.layers: x=layer(x,cp,cq,gp)
        return self.op(x)

    def loss(self,x0,mask,cp_v,cq_v,gamma=CAUCHY_GAMMA):
        """Noise-prediction loss at artificial gap positions + Cauchy regularization."""
        B=x0.shape[0]
        t=torch.randint(1,self.T+1,(B,),device=x0.device)
        ab=self.alpha_bar[t-1].view(B,1,1)
        eps=torch.randn_like(x0)
        x_t=ab.sqrt()*x0+(1-ab).sqrt()*eps
        x_in=x_t*(1-mask)+x0*mask
        pred=self.forward(x_in,t,mask,cp_v,cq_v)
        tgt=(1-mask).bool()
        ld=torch.nn.functional.mse_loss(pred[tgt],eps[tgt])
        lc=self._cauchy()
        return ld+gamma*lc, ld.detach(), lc.detach()

def linear_interp_prior_np(sig,mask):
    out=sig.copy(); L,F=sig.shape
    for f in range(F):
        arr=sig[:,f].copy(); obs=np.where(mask[:,f]>0)[0]
        if len(obs)<2: continue
        gap=np.where(mask[:,f]==0)[0]
        if len(gap)==0: continue
        pre=obs[obs<gap[0]]; post=obs[obs>gap[-1]]
        if len(pre)==0 or len(post)==0: continue
        v0=arr[pre[-1]]; i0=pre[-1]; v1=arr[post[0]]; i1=post[0]
        for i in gap: out[i,f]=v0+(v1-v0)*(i-i0)/(i1-i0)
    return out

def adaptive_T(gl): return int(np.clip(gl*2,20,T_STEPS))

def build_pidck_ctx(pat,gs,ge,dur,pm,ps,qm,qs):
    pf=_exw(pat,gs,CTX_WIN,'pre'); pf['gap_duration_mins']=dur
    post=pat[(pat['ts']>ge)&(pat['ts']<=ge+pd.Timedelta(minutes=CTX_WIN))]
    og=post['GlucoseCGM']; oh=post['HR']; st,ct=_tod(gs)
    post_vals=np.array([og.mean(),og.min() if len(og)>0 else np.nan,
        og.dropna().iloc[0] if og.dropna().size>0 else np.nan,_ms(og),
        int((og<HYPO_THR).any()),int((og<HYPO_THR).sum()),oh.mean(),
        oh.dropna().iloc[0] if oh.dropna().size>0 else np.nan,_ms(oh),st,ct,
        float(dur)],dtype=np.float64)  # gap_duration_mins as 12th post feature
    hf,cf=_hc(pat,gs,pf); ap={}; ap.update(pf); ap.update(hf); ap.update(cf)
    pre_vals=np.array([ap.get(k,np.nan) for k in PRE_COLS],dtype=np.float64)
    pn=np.nan_to_num((pre_vals-pm)/ps,nan=0.).astype(np.float32)
    qn=np.nan_to_num((post_vals-qm)/qs,nan=0.).astype(np.float32)
    pmb=(~np.isnan(pre_vals)).astype(np.float32); qmb=(~np.isnan(post_vals)).astype(np.float32)
    return np.concatenate([pn,pmb,[1.]]),np.concatenate([qn,qmb])


def impute_pidck(masked_df,gap_info,model,pm,ps,qm,qs):
    results={}; model.eval()
    for gap_i,g in enumerate(gap_info):
        pid=g['pid']; gl=g['gap_len']; gs=g['ts_start']; ge=g['ts_end']; dur=gl*5
        pat=masked_df[masked_df['PtID']==pid].sort_values('ts').reset_index(drop=True)
        idx=g['start_idx']; ws=max(0,idx-CTX_WIN//5); we=min(len(pat),idx+gl+CTX_WIN//5)
        seg=pat.iloc[ws:we]; L=len(seg)
        if L>MAX_L: results[id(g)]={'glucose':[np.nan]*gl,'hr':[np.nan]*gl}; continue
        gn=(seg['GlucoseCGM'].values-G_MEAN)/G_STD; hn=(seg['HR'].values-H_MEAN)/H_STD
        sig=np.stack([gn,hn],axis=1); msk=(~np.isnan(sig)).astype(np.float32)
        sig=np.nan_to_num(sig,nan=0.); prior_np=linear_interp_prior_np(sig,msk)
        xi=torch.tensor(sig,dtype=torch.float32).unsqueeze(0).to(DEVICE)
        mt=torch.tensor(msk,dtype=torch.float32).unsqueeze(0).to(DEVICE)
        prior=torch.tensor(prior_np,dtype=torch.float32).unsqueeze(0).to(DEVICE)
        pv,qv=build_pidck_ctx(pat,gs,ge,dur,pm,ps,qm,qs)
        if np.all(pv[:len(PRE_COLS)] == 0):
            warnings.warn(f"PIDCK: all pre context features are zero for gap {g['pid']} {g['ts_start']} — context may not be working")
        cp=torch.tensor(pv,dtype=torch.float32).unsqueeze(0).to(DEVICE)
        cq=torch.tensor(qv,dtype=torch.float32).unsqueeze(0).to(DEVICE)

        local_start = idx - ws
        local_end   = local_start + gl
        gidx = np.arange(local_start, min(local_end, len(seg)))
        T_eff=adaptive_T(gl)
        bnd=torch.zeros(1,L,NF,device=DEVICE)
        if len(gidx)>0:
            for bi in range(min(2,len(gidx))):
                bnd[:,gidx[bi],:]=0.8-bi*0.3
                bnd[:,gidx[-(bi+1)],:]=0.8-bi*0.3

        chunks=[]
        with torch.inference_mode():
            for nbatch in chain_chunks(N_SAMPLES, CHAIN_BATCH_SIZE):
                xi_b=xi.expand(nbatch,-1,-1).contiguous()
                mt_b=mt.expand(nbatch,-1,-1).contiguous()
                prior_b=prior.expand(nbatch,-1,-1).contiguous()
                cp_b=cp.expand(nbatch,-1).contiguous()
                cq_b=cq.expand(nbatch,-1).contiguous()
                bnd_b=bnd.expand(nbatch,-1,-1)

                x=prior_b.clone(); ns=model.alpha_bar[T_eff-1].sqrt()
                x=x+(1-ns)*torch.randn_like(x); x=x*(1-mt_b)+xi_b*mt_b
                for t in reversed(range(1,T_eff+1)):
                    tt=torch.full((nbatch,),t,device=DEVICE,dtype=torch.long)
                    eps=model(x,tt,mt_b,cp_b,cq_b); ab=model.alpha_bar[t-1]
                    abp=model.alpha_bar[t-2] if t>1 else torch.tensor(1.,device=DEVICE)
                    x0=((x-(1-ab).sqrt()*eps)/ab.sqrt()).clamp(-3,3)
                    x=(abp.sqrt()*x0+(1-abp).sqrt()*torch.randn_like(x) if t>1 else x0)
                    if t>T_eff//2:
                        x=x-bnd_b*(t/T_eff)*0.15*(x-prior_b)
                    x=x*(1-mt_b)+xi_b*mt_b
                chunks.append(x.cpu())

        sn=torch.cat(chunks,dim=0).numpy()
        best_idx = select_best_chain(sn, sig, msk, target_start=local_start, target_len=gl)
        best     = sn[best_idx, :, :]
        med_agg  = np.median(sn, axis=0)
        results[id(g)]={
            'glucose':          (best[gidx, 0]*G_STD+G_MEAN).tolist(),
            'hr':               (best[gidx, 1]*H_STD+H_MEAN).tolist(),
            'glucose_med':      (med_agg[gidx, 0]*G_STD+G_MEAN).tolist(),
            'hr_med':           (med_agg[gidx, 1]*H_STD+H_MEAN).tolist(),
            'glucose_samples':  (sn[:,gidx,0]*G_STD+G_MEAN).astype(np.float32),
            'hr_samples':       (sn[:,gidx,1]*H_STD+H_MEAN).astype(np.float32)}
        release_device_cache(gap_index=gap_i)
    return results


# ── Training ─────────────────────────────────────────────────────────
class PIDCKTrainDataset(Dataset):
    """
    Builds fully observed 4-hour windows from TRAIN patients only.
    Artificial contiguous gaps are introduced during __getitem__.
    Context is extracted from the same TRAIN patient's original record,
    outside the artificial gap.
    """
    def __init__(self, df, pm, ps, qm, qs, seq=SEQ_LEN, mr=MASK_RATIO):
        self.seq=seq
        self.mr=mr
        self.pm=np.asarray(pm,dtype=np.float64)
        self.ps=np.asarray(ps,dtype=np.float64)
        self.qm=np.asarray(qm,dtype=np.float64)
        self.qs=np.asarray(qs,dtype=np.float64)

        self.patient_data={
            pid: pat.sort_values('ts').reset_index(drop=True)
            for pid,pat in df.groupby('PtID')
        }

        self.samples=[]
        for pid,pat in self.patient_data.items():
            g=(pat['GlucoseCGM'].values-G_MEAN)/G_STD
            h=(pat['HR'].values-H_MEAN)/H_STD
            sig=np.stack([g,h],axis=1)
            ts=pat['ts'].values
            for i in range(0,len(sig)-seq,seq//2):
                w=sig[i:i+seq]
                if np.isnan(w).any():
                    continue
                self.samples.append((w.copy(),ts[i:i+seq].copy(),pid))
        self.samples=limit_windows(self.samples)
        print(f"  PIDCK training windows: {len(self.samples):,}")

    def __len__(self):
        return len(self.samples)

    def _make_anchored_mask(self):
        n_mask=max(1,int(self.seq*self.mr))
        max_start=self.seq-n_mask-1
        if max_start < 1:
            max_start=1
        start=int(np.random.randint(1,max_start+1))
        end=start+n_mask-1
        return start,end

    def __getitem__(self,idx):
        w,ts_arr,pid=self.samples[idx]
        x=torch.tensor(w,dtype=torch.float32)

        # 50% anchored contiguous block (never touching the window edges),
        # 50% unrestricted contiguous block.
        if np.random.random() < 0.5:
            start_idx,end_idx=self._make_anchored_mask()
        else:
            n_mask=max(1,int(self.seq*self.mr))
            max_start=self.seq-n_mask
            start_idx=int(np.random.randint(0,max_start+1))
            end_idx=start_idx+n_mask-1

        m=torch.ones_like(x)
        m[start_idx:end_idx+1]=0.

        pat=self.patient_data[pid]
        gs=pd.Timestamp(ts_arr[start_idx])
        ge=pd.Timestamp(ts_arr[end_idx])
        dur=(end_idx-start_idx+1)*5

        pc,qc=build_pidck_ctx(
            pat,gs,ge,dur,
            self.pm,self.ps,self.qm,self.qs
        )

        # Boundary-aware loss weights: 1.0 at the gap edge, 0.667 one step in, 0.333 two steps in.
        bnd=torch.zeros(self.seq,dtype=torch.float32)
        for bi in range(start_idx,end_idx+1):
            dist=min(bi-start_idx,end_idx-bi)
            if dist < 3:
                bnd[bi]=1.0-dist/3.0

        return (
            x,m,
            torch.tensor(pc,dtype=torch.float32),
            torch.tensor(qc,dtype=torch.float32),
            bnd
        )


PRE_DIM_FULL, POST_DIM_FULL = len(PRE_COLS)*2 + 1, len(POST_COLS)*2

# Pre-gap vector layout: [normalised PRE_COLS | observed-indicators for PRE_COLS | constant 1].
# HIST_KEEP zeroes every 1/3/7-day history and contrast feature (value AND its indicator), so the
# model sees exactly the immediate pre-gap statistics, the post-gap vector and the signal window.
_is_hist = np.array([c.startswith('hist_') or c.startswith('contrast_') for c in PRE_COLS])
HIST_KEEP = torch.tensor(np.concatenate([~_is_hist, ~_is_hist, [True]]).astype(np.float32), device=DEVICE)
print(f"History ablation masks {int(_is_hist.sum())} of {len(PRE_COLS)} pre-gap features "
      f"({int(_is_hist.sum())*2} of {PRE_DIM_FULL} input dims incl. indicators).")


class FixedPosWeight(nn.Module):
    """Replaces PosWeightNet: pre/post cross-attention always weighted 0.5 / 0.5."""
    def forward(self, x, cp, cq, gpos):
        w = torch.full_like(x[..., :1], 0.5)
        return w, w

def build_pidck(pin_pw=False):
    model = PIDCKModel(PRE_DIM_FULL, POST_DIM_FULL)
    if pin_pw:
        for layer in model.layers:
            layer.pw = FixedPosWeight()
    return model.to(DEVICE)


def train_pidck(df, ckpt, zero_ctx=False, pin_pw=False, no_hist=False,
                bnd_w=BOUNDARY_W, gamma=CAUCHY_GAMMA, seed=SPLIT_SEED, tag='CAKGI'):
    """Train CAKGI on TRAIN patients only (75 epochs, Adam 1e-3, batch 32, 5-epoch warm-up +
    cosine, gradient clipping 1, diffusion loss + 0.1·Cauchy + 0.3·boundary loss).
    The keyword arguments switch off exactly one ingredient for the ablations (Part B);
    with the defaults this is the full CAKGI model."""
    model = build_pidck(pin_pw)
    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE)); model.eval()
        print(f"  {tag}: loaded {os.path.basename(ckpt)}"); return model
    np.random.seed(seed); torch.manual_seed(seed)
    ds = PIDCKTrainDataset(df, pm_arr, ps_arr, qm_arr, qs_arr)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = lr_schedule(opt)
    print(f"  {tag}: training {EPOCHS} epochs "
          f"(zero_ctx={zero_ctx}, pin_pw={pin_pw}, no_hist={no_hist}, bnd_w={bnd_w}, gamma={gamma}, seed={seed})")
    model.train()
    for ep in range(EPOCHS):
        tot = 0.0
        for x, m, pc, qc, bnd in loader:
            x, m, bnd = x.to(DEVICE), m.to(DEVICE), bnd.to(DEVICE)
            pc, qc = pc.to(DEVICE), qc.to(DEVICE)
            if zero_ctx:
                pc, qc = torch.zeros_like(pc), torch.zeros_like(qc)
            if no_hist:
                pc = pc * HIST_KEEP
            opt.zero_grad()
            total, _, _ = model.loss(x, m, pc, qc, gamma=gamma)
            if bnd_w > 0:
                B = x.shape[0]
                t_b = torch.randint(1, model.T + 1, (B,), device=DEVICE)
                ab_b = model.alpha_bar[t_b-1].view(B, 1, 1); eps_b = torch.randn_like(x)
                xin_b = (ab_b.sqrt()*x + (1-ab_b).sqrt()*eps_b)*(1-m) + x*m
                pred_b = model(xin_b, t_b, m, pc, qc)
                w = bnd.unsqueeze(-1).expand_as(eps_b)*(1-m)
                total = total + bnd_w*(((pred_b - eps_b)**2)*w).sum()/w.sum().clamp(min=1)
            total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += float(total.detach().cpu())
        sched(ep)
        if log_epoch(ep):
            print(f"    {tag} Ep {ep+1:>3}/{EPOCHS} | {tot/len(loader):.4f}")
    torch.save(model.state_dict(), ckpt); model.eval()
    print(f"  {tag}: saved {os.path.basename(ckpt)}")
    return model

print("CAKGI ready.")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 11 — Train (or load) every model on TRAIN patients only
# ════════════════════════════════════════════════════════════════════
print(f"Training patients: {len(TRAIN_PIDS)} | held-out MC patients: {len(TEST_PIDS)} | epochs: {EPOCHS}")

# ── CAKGI context normalisation, fitted on TRAIN patients' context vectors only ──
vdf = pd.read_csv(VECTORS_FILE)
pid_col = next((c for c in ['PtID', 'pid', 'patient_id', 'PatientID'] if c in vdf.columns), None)
if pid_col is None:
    raise RuntimeError("VECTORS_FILE has no patient-ID column; it is needed for train-only normalisation.")
vdf_train = vdf[vdf[pid_col].astype(str).isin(set(map(str, TRAIN_PIDS)))]
if len(vdf_train) == 0:
    raise RuntimeError("No context-vector rows matched TRAIN_PIDS; check the PtID format.")
missing = [c for c in PRE_COLS + POST_COLS if c not in vdf_train.columns]
if missing:
    raise RuntimeError(f"Missing context columns: {missing}")
pm_arr = np.nan_to_num(vdf_train[PRE_COLS].mean().values.astype(np.float64), nan=0.0)
ps_arr = vdf_train[PRE_COLS].std().replace(0, 1).fillna(1).values.astype(np.float64)
qm_arr = np.nan_to_num(vdf_train[POST_COLS].mean().values.astype(np.float64), nan=0.0)
qs_arr = vdf_train[POST_COLS].std().replace(0, 1).fillna(1).values.astype(np.float64)
print(f"  context normalisation from {len(vdf_train):,} TRAIN-patient vectors "
      f"(PRE {len(PRE_COLS)} -> {PRE_DIM_FULL} dims, POST {len(POST_COLS)} -> {POST_DIM_FULL} dims)")

# ── Models ────────────────────────────────────────────────────────────
t0 = time.perf_counter()
model_brits = train_brits_faithful(train_df)
model_gpvae = train_gpvae_banded(train_df)
model_catsi = train_catsi_official(train_df)
model_csdi  = train_csdi(train_df)
model_pidck = train_pidck(train_df, PIDCK_MODEL)
print(f"All models ready ({(time.perf_counter()-t0)/60:.1f} min).")

# ── Leakage checks: changing the hidden (masked) values must not change any output ──
with torch.no_grad():
    torch.manual_seed(0)
    _x, _m = next(iter(DataLoader(CGMDatasetSeq(train_df), batch_size=8, shuffle=True)))
    _x, _m = _x.to(DEVICE), _m.to(DEVICE)
    _x2 = _x + (1 - _m) * 5.0
    _d_brits = (model_brits(_x, _m) - model_brits(_x2, _m)).abs().max().item()
    _d_gpvae = (model_gpvae.encode(_x, _m)[0] - model_gpvae.encode(_x2, _m)[0]).abs().max().item()
assert _d_brits < 1e-5, f"BRITS output depends on hidden values ({_d_brits})"
assert _d_gpvae < 1e-5, f"GP-VAE encoder depends on hidden values ({_d_gpvae})"
assert set(map(str, TRAIN_PIDS)).isdisjoint(set(map(str, TEST_PIDS)))
_ck_pidck = torch.load(PIDCK_MODEL, map_location='cpu')
assert _ck_pidck['pe.net.0.weight'].shape[1] == PRE_DIM_FULL and _ck_pidck['qe.net.0.weight'].shape[1] == POST_DIM_FULL
print("Leakage checks passed: models fit on TRAIN patients only; BRITS / GP-VAE ignore hidden values.")
# CSDI-adapted and CAKGI zero the masked positions before the network (x·mask, or noise at
# the masked positions); CATSI and MICE receive NaN at the masked positions.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 12 — Monte Carlo evaluation: 3 trials × N_GAPS independent gaps, every method
#           (resumable: results are saved after every trial × method)
# ════════════════════════════════════════════════════════════════════
_EVAL_PATIENT_CACHE = {pid: pat.sort_values('ts').reset_index(drop=True)
                       for pid, pat in test_df.groupby('PtID')}

def _masked_patient_for_gap(g):
    """The held-out patient's record with ONLY this sample's target interval hidden."""
    pat = _EVAL_PATIENT_CACHE[g['pid']].copy()
    idx = int(g['start_idx']); gl = int(g['gap_len'])
    cols = [pat.columns.get_loc('GlucoseCGM'), pat.columns.get_loc('HR')]
    pat.iloc[idx:idx+gl, cols] = np.nan
    return pat

def impute_independent(method_fn, gap_info, *args):
    """Evaluate every Monte Carlo sample with only that sample masked."""
    merged = {}; n = len(gap_info)
    for j, g in enumerate(gap_info):
        one = method_fn(_masked_patient_for_gap(g), [g], *args)
        if id(g) in one:
            merged[id(g)] = one[id(g)]
        if (j + 1) % 100 == 0 or (j + 1) == n:
            print(f"      {j+1}/{n} independent gaps done", flush=True)
    return merged

def gap_key(g):
    return f"{g['pid']}|{int(g['start_idx'])}|{int(g['gap_len'])}"

def rows_for(method, trial, seed, g, res):
    """One row per (signal, protocol). 'median' uses the median of chains when the method
    provides it, otherwise the point estimate; CRPS uses the samples when provided."""
    out = []
    for signal, gtk in [('glucose', 'gt_glucose'), ('hr', 'gt_hr')]:
        ens = res.get(f'{signal}_samples')
        for agg in ['best', 'median']:
            pv = res.get(f'{signal}_med', res[signal]) if agg == 'median' else res[signal]
            m = compute_metrics(g[gtk], pv, samples=ens)
            out.append(dict(trial=trial, seed=seed, method=method, agg=agg, signal=signal,
                            gap_len=g['gap_len'], gap_len_min=g['gap_len_min'],
                            pid=str(g['pid']), gap_key=gap_key(g), **m))
    return out

def run_arms(arms, out_csv, trials=(1, 2, 3)):
    """Run {name: imputer(masked_df, gap_info)} on the SAME gaps of every trial; resumable."""
    rows = pd.read_csv(out_csv).to_dict('records') if os.path.exists(out_csv) else []
    done = {(int(r['trial']), r['method']) for r in rows}
    for t in trials:
        seed, gaps = RANDOM_SEEDS[t-1], GAPS_BY_TRIAL[t]
        for name, fn in arms.items():
            if (t, name) in done:
                print(f"  trial {t} {name}: already done, skipping"); continue
            torch.manual_seed(10_000*t + seed); np.random.seed(10_000*t + seed)
            t0 = time.perf_counter(); print(f"  trial {t}: {name}...", flush=True)
            res = impute_independent(fn, gaps)
            for g in gaps:
                if id(g) in res:
                    rows += rows_for(name, t, seed, g, res[id(g)])
            pd.DataFrame(rows).to_csv(out_csv, index=False)
            done.add((t, name))
            print(f"    {name} finished in {(time.perf_counter()-t0)/60:.1f} min", flush=True)
            release_device_cache(force=True)
    df = pd.DataFrame(rows); df['pid'] = df['pid'].astype(str)
    return df


# ── The gap sets (identical for every method; regenerated from the trial seeds) ──
GAPS_BY_TRIAL = {}
for t, seed in enumerate(RANDOM_SEEDS[:N_TRIALS], start=1):
    _, gi = simulate_gaps(test_df, N_GAPS, GAP_DURATIONS, GAP_PROBS, np.random.default_rng(seed))
    assert {str(g['pid']) for g in gi} <= set(map(str, TEST_PIDS))     # held-out patients only
    GAPS_BY_TRIAL[t] = gi

MAIN_ARMS = {
    'mice' : impute_mice,
    'brits': lambda d, gi: impute_brits(d, gi, model_brits),
    'catsi': lambda d, gi: impute_catsi(d, gi, model_catsi),
    'gpvae': lambda d, gi: impute_gpvae_banded(d, gi, model_gpvae),
    'csdi' : lambda d, gi: impute_csdi(d, gi, model_csdi),
    'pidck': lambda d, gi: impute_pidck(d, gi, model_pidck, pm_arr, ps_arr, qm_arr, qs_arr),
}
main_df = run_arms(MAIN_ARMS, MAIN_RESULTS, trials=range(1, N_TRIALS + 1))
print(f"\nMain results: {len(main_df):,} rows -> {MAIN_RESULTS}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 13 — Table 1 (mean ± SD across gaps) and RMSE by gap duration
#   Needs only Cell 1 and a results CSV (MAIN_RESULTS or the released ALL_RESULTS).
# ════════════════════════════════════════════════════════════════════
TABLE_SRC = ALL_RESULTS if os.path.exists(ALL_RESULTS) else MAIN_RESULTS
R = pd.read_csv(TABLE_SRC)
print(f"Results: {os.path.basename(TABLE_SRC)} | {len(R):,} rows | "
      f"{R['trial'].nunique()} trials | {R.drop_duplicates(['trial', 'gap_key']).shape[0]:,} gaps")

SIGS, METS = ['glucose', 'hr'], ['rmse', 'mae', 'crps']
GROUPS = [[('mice', 'MICE', False), ('brits', 'BRITS', False), ('catsi', 'CATSI', False),
           ('gpvae', 'GP-VAE', False)],
          [('csdi', 'CSDI-adapted', True), ('pidck', r'\textbf{CAKGI}$^\dagger$', True)]]

def stat(method, agg, sig, met):
    v = R[(R['method'] == method) & (R['agg'] == agg) & (R['signal'] == sig)][met].dropna()
    return (v.mean(), v.std()) if len(v) else (np.nan, np.nan)

# ── readable summary ──
rows = []
for key, label, two in [r for g in GROUPS for r in g]:
    for agg in (['median', 'best'] if two else ['median']):
        row = {'method': METHOD_LABELS[key], 'protocol': ('best-chain' if agg == 'best' else
                                                          'median' if two else '—')}
        for s in SIGS:
            for m in METS:
                mu, sd = stat(key, agg, s, m)
                row[f'{s} {m.upper()}'] = f"{mu:.2f} ± {sd:.2f}"
        rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(f"{RESULTS_DIR}/table_mc_main.csv", index=False)
with pd.option_context('display.width', 250, 'display.max_columns', 20):
    print(summary.to_string(index=False))

# ── LaTeX (Table 1) ──
cells = {}
for grp in GROUPS:
    for key, _, two in grp:
        for agg in (['best', 'median'] if two else ['median']):
            for s in SIGS:
                for m in METS:
                    cells[(key, agg, s, m)] = stat(key, agg, s, m)
best = {(s, m): min(v[0] for k, v in cells.items() if k[2] == s and k[3] == m and np.isfinite(v[0]))
        for s in SIGS for m in METS}

def fmt(key, agg, s, m):
    mu, sd = cells[(key, agg, s, m)]
    if not np.isfinite(mu):
        return 'N/A'
    txt = f"{mu:.2f}{{\\pm}}{sd:.2f}"
    return f"$\\mathbf{{{txt}}}$" if abs(mu - best[(s, m)]) < 5e-3 else f"${txt}$"

L = [r"\begin{table*}[t]", r"\centering", r"\scriptsize",
     r"\caption{Monte Carlo evaluation on held-out patients ($N{=}500$ independent gaps ${\times}\,3$ trials, "
     r"50 chains for probabilistic methods). Values are mean ${\pm}$ SD across gaps; CRPS equals MAE for "
     r"deterministic methods. Best value per column in bold.}",
     r"\label{tab:mc_main}", r"\setlength{\tabcolsep}{1.5pt}",
     r"\begin{tabular}{@{}llcccccc@{}}", r"\toprule",
     r"& & \multicolumn{3}{c}{\textbf{Glucose} (mg/dL)} & \multicolumn{3}{c}{\textbf{HR} (bpm)} \\",
     r"\cmidrule(lr){3-5}\cmidrule(lr){6-8}",
     r"\textbf{Method} & \textbf{Protocol} & \textbf{RMSE} & \textbf{MAE} & \textbf{CRPS} "
     r"& \textbf{RMSE} & \textbf{MAE} & \textbf{CRPS} \\", r"\midrule"]
for gi, grp in enumerate(GROUPS):
    if gi:
        L.append(r"\midrule")
    for key, label, two in grp:
        if not two:
            L.append(' & '.join([label, '---'] + [fmt(key, 'median', s, m) for s in SIGS for m in METS]) + r' \\')
        else:
            for j, (agg, prot) in enumerate([('best', 'Best-chain'), ('median', 'Median')]):
                row = [rf"\multirow{{2}}{{*}}{{{label}}}" if j == 0 else '', prot]
                for s in SIGS:
                    row += [fmt(key, agg, s, 'rmse'), fmt(key, agg, s, 'mae')]
                    row += [rf"\multirow{{2}}{{*}}{{{fmt(key, 'median', s, 'crps')}}}" if j == 0 else '']
                L.append(' & '.join(row) + r' \\')
L += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
latex_main = '\n'.join(L)
open(f"{RESULTS_DIR}/table_mc_main.tex", 'w').write(latex_main)
print('\n' + latex_main + f"\n\nSaved -> {RESULTS_DIR}/table_mc_main.tex")

# ── RMSE by gap duration (median-of-chains; also best-chain for the diffusion models) ──
dur_tabs = []
for signal in SIGS:
    for agg in ['median', 'best']:
        s = R[(R['agg'] == agg) & (R['signal'] == signal) & (R['method'].isin(MAIN_METHODS))]
        if agg == 'best':
            s = s[s['method'].isin(['csdi', 'pidck'])]
        d = s.pivot_table(index='method', columns='gap_len_min', values='rmse', aggfunc='mean')
        d = d.reindex([m for m in MAIN_METHODS if m in d.index]).round(2)
        d.index = [METHOD_LABELS[m] for m in d.index]
        print(f"\n--- {signal.upper()} RMSE by gap duration (min) [{agg}] ---\n{d.to_string()}")
        dur_tabs.append(d.assign(signal=signal, protocol=agg))
pd.concat(dur_tabs).to_csv(f"{RESULTS_DIR}/table_rmse_by_duration.csv")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 14 — Figures
#   (a) RMSE vs gap duration and RMSE distributions (from the results CSV; median of chains)
#   (b) example imputations: the first trial-1 gap of every duration, imputed by every
#       method (needs the trained models from Cell 11 and the gaps from Cell 12)
# ════════════════════════════════════════════════════════════════════
R = pd.read_csv(ALL_RESULTS if os.path.exists(ALL_RESULTS) else MAIN_RESULTS)
med = R[(R['agg'] == 'median') & (R['method'].isin(MAIN_METHODS))]
saved = []

for signal, lab, unit in [('glucose', 'Glucose', 'mg/dL'), ('hr', 'HR', 'bpm')]:
    s = med[med['signal'] == signal]
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for m in MAIN_METHODS:
        d = s[s['method'] == m].groupby('gap_len_min')['rmse'].mean()
        if len(d):
            ax.plot(d.index / 60, d.values, marker='o', lw=1.8, color=COLORS[m], label=METHOD_LABELS[m])
    ax.set_xlabel('Gap duration (h)'); ax.set_ylabel(f'RMSE ({unit})')
    ax.set_title(f'{lab}: RMSE by gap duration (median of chains)'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
    fig.tight_layout(); p = f"{PLOT_DIR}/rmse_vs_duration_{signal}.png"; fig.savefig(p, dpi=200); plt.close(fig); saved.append(p)

    fig, ax = plt.subplots(figsize=(7, 4.2))
    data = [s[s['method'] == m]['rmse'].dropna().values for m in MAIN_METHODS]
    bp = ax.boxplot(data, patch_artist=True, medianprops=dict(color='black', lw=1.5),
                    flierprops=dict(marker='o', markersize=2, alpha=0.3))
    for patch, m in zip(bp['boxes'], MAIN_METHODS):
        patch.set_facecolor(COLORS[m]); patch.set_alpha(0.75)
    ax.set_xticks(range(1, len(MAIN_METHODS) + 1))
    ax.set_xticklabels([METHOD_LABELS[m] for m in MAIN_METHODS], rotation=20, ha='right')
    ax.set_ylabel(f'RMSE per gap ({unit})'); ax.set_title(f'{lab}: RMSE distribution'); ax.grid(axis='y', alpha=0.3)
    fig.tight_layout(); p = f"{PLOT_DIR}/rmse_box_{signal}.png"; fig.savefig(p, dpi=200); plt.close(fig); saved.append(p)

# ── (b) example imputations ──
if 'GAPS_BY_TRIAL' in globals() and 'model_pidck' in globals():
    examples = {}
    for g in GAPS_BY_TRIAL[1]:
        examples.setdefault(g['gap_len_min'], g)
    torch.manual_seed(0); np.random.seed(0)
    for dur_min in sorted(examples):
        g = examples[dur_min]
        preds = {m: impute_independent(fn, [g]).get(id(g), {}) for m, fn in MAIN_ARMS.items()}
        pat = _EVAL_PATIENT_CACHE[g['pid']]
        idx, gl = int(g['start_idx']), int(g['gap_len'])
        cs, ce = max(0, idx - 48), min(len(pat), idx + gl + 48)          # 4 h either side
        t_all = (np.arange(cs, ce) - idx) * 5 / 60; t_gap = np.arange(gl) * 5 / 60
        for signal, col, gtk, lab, unit in [('glucose', 'GlucoseCGM', 'gt_glucose', 'Glucose', 'mg/dL'),
                                            ('hr', 'HR', 'gt_hr', 'HR', 'bpm')]:
            y = pat[col].values[cs:ce].astype(float).copy(); y[idx-cs:idx-cs+gl] = np.nan
            fig, ax = plt.subplots(figsize=(10, 4))
            ax.plot(t_all, y, 'k-', lw=2, label='Observed')
            ax.plot(t_gap, g[gtk], 'k--', lw=1.8, label='Ground truth')
            ax.axvspan(0, (gl - 1) * 5 / 60, color='gray', alpha=0.1)
            for m in MAIN_METHODS:
                v = preds[m].get(f'{signal}_med', preds[m].get(signal))
                if v is not None and len(v) == gl:
                    ax.plot(t_gap, v, color=COLORS[m], lw=1.6, alpha=0.9, label=METHOD_LABELS[m])
            ax.set_xlabel('Time from gap start (h)'); ax.set_ylabel(f'{lab} ({unit})')
            ax.set_title(f'{lab}: {dur_min // 60}-h gap (held-out patient; median of chains for diffusion models)')
            ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=4)
            fig.tight_layout(); p = f"{PLOT_DIR}/example_{signal}_{dur_min}min.png"
            fig.savefig(p, dpi=200); plt.close(fig); saved.append(p)
else:
    print("Example imputations skipped (run Cells 2–12 first).")

print(f"Saved {len(saved)} figures to {PLOT_DIR}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 15 — Paired statistics: CAKGI vs every baseline
#   Paired per-gap differences (CAKGI − baseline; < 0 = CAKGI better) on the SAME gaps.
#   pat_ci: 95 % CI from a patient-level cluster bootstrap (resample held-out patients with
#   all their gaps, 10,000 resamples) — the CI reported in the paper. gap_ci: naive per-gap
#   bootstrap, for reference. patients_better: patients whose mean difference is < 0;
#   wilcoxon_p: Wilcoxon signed-rank test on the per-patient mean differences.
#   Needs only Cell 1 and a results CSV.
# ════════════════════════════════════════════════════════════════════
def paired_patient_bootstrap(df, ours, ref, agg='median', n_boot=10_000, seed=0,
                             signals=('glucose', 'hr'), metrics=('rmse', 'mae', 'crps')):
    rng, rows = np.random.default_rng(seed), []
    for signal in signals:
        s = df[(df['agg'] == agg) & (df['signal'] == signal)]
        a = s[s['method'] == ours].set_index(['trial', 'gap_key'])
        b = s[s['method'] == ref].set_index(['trial', 'gap_key'])
        a, b = a[~a.index.duplicated()], b[~b.index.duplicated()]
        common = a.index.intersection(b.index)
        for m in metrics:
            d = a.loc[common, m] - b.loc[common, m]; ok = d.notna()
            d, pid = d[ok].to_numpy(float), a.loc[common, 'pid'][ok].astype(str).to_numpy()
            if len(d) < 10:
                continue
            pids = np.unique(pid); P = len(pids)
            sums = np.array([d[pid == p].sum() for p in pids])
            cnts = np.array([(pid == p).sum() for p in pids])
            pmean = sums / cnts
            draw = rng.integers(0, P, size=(n_boot, P))            # resample PATIENTS
            boot_gw = sums[draw].sum(1) / cnts[draw].sum(1)         # gap-weighted mean
            boot_pw = pmean[draw].mean(1)                           # each patient weighted equally
            gi = rng.integers(0, len(d), size=(n_boot, len(d)))     # naive gap-level bootstrap
            boot_gap = d[gi].mean(1)
            n_better = int((pmean < 0).sum())
            rows.append(dict(
                ours=ours, baseline=ref, agg=agg, signal=signal, metric=m,
                n_gaps=len(d), n_patients=P, mean_diff=d.mean(),
                gap_ci_lo=np.percentile(boot_gap, 2.5), gap_ci_hi=np.percentile(boot_gap, 97.5),
                pat_ci_lo=np.percentile(boot_gw, 2.5),  pat_ci_hi=np.percentile(boot_gw, 97.5),
                patmean_diff=pmean.mean(),
                patmean_ci_lo=np.percentile(boot_pw, 2.5), patmean_ci_hi=np.percentile(boot_pw, 97.5),
                patients_better=f"{n_better}/{P}",
                wilcoxon_p=wilcoxon(pmean).pvalue if P >= 6 and np.any(pmean != 0) else np.nan,
                sign_p=binomtest(n_better, P, 0.5).pvalue,
                gap_win_rate=float((d < 0).mean()),
                pat_significant=bool(np.percentile(boot_gw, 97.5) < 0 or np.percentile(boot_gw, 2.5) > 0)))
    return rows


R = pd.read_csv(ALL_RESULTS if os.path.exists(ALL_RESULTS) else MAIN_RESULTS)
pb_main = pd.DataFrame([r for ref in ['csdi', 'brits', 'catsi', 'gpvae', 'mice'] if ref in set(R['method'])
                        for agg in ['median', 'best']
                        for r in paired_patient_bootstrap(R, 'pidck', ref, agg=agg)])
pb_main.to_csv(f"{RESULTS_DIR}/paired_bootstrap_cakgi_vs_baselines.csv", index=False)
show = ['baseline', 'agg', 'signal', 'metric', 'n_gaps', 'n_patients', 'mean_diff',
        'pat_ci_lo', 'pat_ci_hi', 'patients_better', 'wilcoxon_p', 'gap_ci_lo', 'gap_ci_hi']
with pd.option_context('display.width', 250, 'display.max_rows', 500):
    print(pb_main[pb_main['agg'] == 'median'][show].round(3).to_string(index=False))
print(f"\nSaved -> {RESULTS_DIR}/paired_bootstrap_cakgi_vs_baselines.csv "
      f"(median and best-chain protocols; deterministic baselines have identical rows for both).")

## Part B — Ablations and context baselines

Every arm runs on the **same 3 × 500 gaps** as Part A and is compared with the CAKGI reference arm (`pidck_ref`) using the patient-level bootstrap from Cell 15.

**Retrained CAKGI variants.** Each is trained for 75 epochs with the same data, masking, optimiser and schedule as CAKGI; exactly one ingredient changes.
- `pidck_noctx`: all context removed (pre and post vectors zeroed).
- `pidck_nohist`: the 1/3/7-day history and contrast features removed.
- `pidck_pinpw`: the learned position weights fixed at 0.5 / 0.5.
- `pidck_no_bndloss`: the boundary-aware loss removed.
- `pidck_no_cauchy`: the Cauchy regulariser removed.
- `pidck_reseed`: full CAKGI retrained with a second seed. This gives the retraining noise floor.

**Inference recipe.** `pidck_noise` is the same CAKGI checkpoint run as plain diffusion: noise initialisation, full T = 100 and no boundary guidance.

**CSDI-adapted with CAKGI's recipe.**
- `csdi_interp`: CSDI-adapted with CAKGI's inference recipe (interpolation initialisation, adaptive T, boundary guidance).
- `csdi_bnd`: CSDI-adapted retrained for 75 epochs with CAKGI's boundary-aware loss and masking regime, then run with the same recipe.

**Context baselines.**
- `tod_raw`: for each target step, the median of the patient's own values at the same clock time on each of the previous 7 days.
- `ctx_ridge_cakgi`: a linear interpolation anchor plus a ridge regression of the residual on CAKGI's own context vector. It is trained on 2,000 synthetic gaps from the training patients.

**Reference arm.** `pidck_ref` is CAKGI run through the shared sampler below. Its interpolation prior bridges every missing run in the window with that run's own neighbours. The Part A implementation drew one line across all missing indices of the window, which differs only when the window contains other natural gaps. The two runs differ by at most 0.05 in RMSE (30.26 vs 30.25 mg/dL; 14.90 vs 14.95 bpm).

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 16 — Part B setup: interpolation prior, shared gap-position encoding, context cache
# ════════════════════════════════════════════════════════════════════
def linear_interp_runs_np(sig, mask):
    """
    Per-run linear interpolation in normalised space.

    Corrects linear_interp_prior_np, which took ALL missing indices in the window as one
    gap and drew a single line from before the FIRST missing value to after the LAST one.
    With natural missingness elsewhere in the context window (common for HR), the synthetic
    gap was interpolated between distant points; if the window began or ended with a
    missing value, no prior was written at all (left at 0 = training mean).
    np.interp bridges every missing run with its own neighbours.
    """
    out = sig.copy()
    L, nf = sig.shape
    pos = np.arange(L)
    for f in range(nf):
        obs = mask[:, f] > 0
        if obs.sum() == 0 or obs.all():
            continue
        out[~obs, f] = np.interp(pos[~obs], pos[obs], sig[obs, f])
    return out

PRIOR_FN = linear_interp_runs_np   # prior used by every Part B diffusion arm


# All chains share one mask at inference: compute gap positions once and broadcast.
# Output is identical; removes a per-chain Python loop at every diffusion step.
if not hasattr(PIDCKModel, '_gpos_orig'):          # keep the true original across re-runs
    PIDCKModel._gpos_orig = PIDCKModel._gpos

def _gpos_shared(self, mask, B, L):
    if B > 1 and bool((mask == mask[:1]).all()):
        return PIDCKModel._gpos_orig(self, mask[:1], 1, L).repeat(B, 1, 1)  # real copy: MPS-safe
    return PIDCKModel._gpos_orig(self, mask, B, L)

PIDCKModel._gpos = _gpos_shared                    # re-running this cell always installs this version
PIDCKModel._gpos_patched = True


# Context vectors are identical for a gap across every CAKGI arm (the synthetic gap is
# outside all context windows), so build each one once and reuse it.
_CTX_CACHE = {}
if '_build_ctx_uncached' not in globals():
    _build_ctx_uncached = build_pidck_ctx

def build_pidck_ctx(pat, gs, ge, dur, pm, ps, qm, qs):
    k = (str(pat['PtID'].iloc[0]), pd.Timestamp(gs), pd.Timestamp(ge), int(dur))
    if k not in _CTX_CACHE:
        _CTX_CACHE[k] = _build_ctx_uncached(pat, gs, ge, dur, pm, ps, qm, qs)
    return _CTX_CACHE[k]

print("Part B setup ready.")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 17 — Context baselines: time-of-day profile and ridge on CAKGI's context vector
# ════════════════════════════════════════════════════════════════════
def _patient(masked_df, g):
    return masked_df[masked_df['PtID'] == g['pid']].sort_values('ts').reset_index(drop=True)


TOD_DAYS = 7
TOD_TOL  = pd.Timedelta(minutes=10)
TOD_FALLBACK = {'n': 0, 'total': 0}   # target steps with no prior-day history -> linear

def _tod_profile(s_obs, ts_q):
    vals = []
    for d in range(1, TOD_DAYS + 1):
        q = ts_q - pd.Timedelta(days=d)
        vals.append(s_obs.reindex(q, method='nearest', tolerance=TOD_TOL).to_numpy(float))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return np.nanmedian(np.vstack(vals), axis=0)

def impute_tod(masked_df, gap_info, anchored=True):
    """Median of the patient's own values at the same clock time on each of the previous
    7 days (prior days only). anchored=True adds an offset ramp so the profile meets the
    observed values on either side of the gap."""
    res = {}
    for g in gap_info:
        pat = _patient(masked_df, g); idx, gl = int(g['start_idx']), int(g['gap_len'])
        ts = pd.DatetimeIndex(pat['ts'])
        out = {}
        for sig, col in [('glucose', 'GlucoseCGM'), ('hr', 'HR')]:
            y = pat[col].to_numpy(float); pos = np.arange(len(y)); ob = np.isfinite(y)
            if not ob.any():
                out[sig] = [np.nan]*gl; continue
            lin = np.interp(pos, pos[ob], y[ob])
            s = pd.Series(y[ob], index=ts[ob]); s = s[~s.index.duplicated()]
            pred = _tod_profile(s, ts[idx:idx+gl])
            if anchored:
                pre_i = pos[ob & (pos < idx)]; post_i = pos[ob & (pos >= idx + gl)]
                d0 = d1 = np.nan
                if len(pre_i):
                    d0 = y[pre_i[-1]] - _tod_profile(s, ts[[pre_i[-1]]])[0]
                if len(post_i):
                    d1 = y[post_i[0]] - _tod_profile(s, ts[[post_i[0]]])[0]
                u = np.arange(1, gl + 1) / (gl + 1)
                if np.isfinite(d0) and np.isfinite(d1): off = (1-u)*d0 + u*d1
                elif np.isfinite(d0):                   off = d0
                elif np.isfinite(d1):                   off = d1
                else:                                   off = 0.0
                pred = pred + off
            miss = ~np.isfinite(pred)
            TOD_FALLBACK['n'] += int(miss.sum()); TOD_FALLBACK['total'] += gl
            pred[miss] = lin[idx:idx+gl][miss]
            out[sig] = pred.tolist()
        res[id(g)] = out
    return res


W_CTX = CTX_WIN // 5

def _win_stats(w, from_end):
    pos = np.arange(len(w)); ok = np.isfinite(w)
    if ok.sum() == 0:
        return [np.nan]*4
    v, p = w[ok], pos[ok]
    edge = v[-1] if from_end else v[0]
    slope = np.polyfit(p, v, 1)[0] if ok.sum() >= 2 else np.nan
    return [v.mean(), edge, slope, v.std()]

def _gap_features(G, H, ts, idx, gl):
    """Uses ONLY the pre/post windows; values inside the gap are never read."""
    sg = _win_stats(G[max(0, idx-W_CTX):idx], True) + _win_stats(G[idx+gl:idx+gl+W_CTX], False)
    sh = _win_stats(H[max(0, idx-W_CTX):idx], True) + _win_stats(H[idx+gl:idx+gl+W_CTX], False)
    st, ct = _tod(pd.Timestamp(ts[idx]))
    u = np.arange(1, gl + 1) / (gl + 1)
    base = np.column_stack([u, u*(1-u), np.full(gl, gl*5.0), np.full(gl, st), np.full(gl, ct)])
    X = np.hstack([base, np.tile(np.array(sg + sh, float), (gl, 1))])
    return X, base, sg, sh, u

def _anchor(stats, u, fallback):
    a, b = stats[1], stats[5]          # pre_last, post_first
    if np.isfinite(a) and np.isfinite(b): return (1-u)*a + u*b
    if np.isfinite(a): return np.full_like(u, a)
    if np.isfinite(b): return np.full_like(u, b)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore'); m = np.nanmean([stats[0], stats[4]])
    return np.full_like(u, m if np.isfinite(m) else fallback)

def _ridge_design(pat, g, kind):
    """Per-step design matrix for one gap. kind='sum' -> hand summaries;
    kind='cakgi' -> CAKGI's own normalised pre/post context vector."""
    idx, gl = int(g['start_idx']), int(g['gap_len'])
    G = pat['GlucoseCGM'].to_numpy(float); H = pat['HR'].to_numpy(float)
    X, base, sg, sh, u = _gap_features(G, H, pat['ts'].values, idx, gl)
    if kind == 'cakgi':
        pv, qv = build_pidck_ctx(pat, g['ts_start'], g['ts_end'], gl*5,
                                 pm_arr, ps_arr, qm_arr, qs_arr)
        X = np.hstack([base, np.tile(np.concatenate([pv, qv]), (gl, 1))])
    ag, ah = _anchor(sg, u, G_MEAN), _anchor(sh, u, H_MEAN)
    return np.column_stack([X, ag]), np.column_stack([X, ah]), ag, ah, G, H

def train_ctx_ridges(df, kinds, n_gaps=RIDGE_TRAIN_GAPS, seed=SPLIT_SEED):
    """Linear anchor + ridge residual correction. TRAIN patients only; synthetic gaps
    drawn with the same duration mix as the MC benchmark."""
    _, gaps = simulate_gaps(df, n_gaps, GAP_DURATIONS, GAP_PROBS, np.random.default_rng(seed))
    cache = {pid: p.sort_values('ts').reset_index(drop=True) for pid, p in df.groupby('PtID')}
    mk = lambda: make_pipeline(SimpleImputer(strategy='median', add_indicator=True),
                               StandardScaler(), Ridge(alpha=1.0))
    models = {}
    for kind in kinds:
        XG, YG, XH, YH = [], [], [], []
        for g in gaps:
            p = cache[g['pid']]; idx, gl = int(g['start_idx']), int(g['gap_len'])
            xg, xh, ag, ah, G, H = _ridge_design(p, g, kind)
            XG.append(xg); YG.append(G[idx:idx+gl] - ag)
            yh = H[idx:idx+gl]; ok = np.isfinite(yh)
            if ok.any():
                XH.append(xh[ok]); YH.append((yh - ah)[ok])
        models[kind] = (mk().fit(np.vstack(XG), np.concatenate(YG)),
                        mk().fit(np.vstack(XH), np.concatenate(YH)))
        print(f"  ridge[{kind}] trained on {len(gaps)} TRAIN gaps, {np.vstack(XG).shape[1]} features")
    return models

def make_ridge_imputer(kind):
    def _imp(masked_df, gap_info):
        mg, mh = RIDGES[kind]; res = {}
        for g in gap_info:
            pat = _patient(masked_df, g)
            xg, xh, ag, ah, _, _ = _ridge_design(pat, g, kind)
            res[id(g)] = {'glucose': (ag + mg.predict(xg)).tolist(),
                          'hr':      (ah + mh.predict(xh)).tolist()}
        return res
    return _imp

RIDGES = train_ctx_ridges(train_df, ['cakgi'])

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 18 — One diffusion sampler for every Part B diffusion arm
#   init ('interp' | 'noise'), schedule ('adaptive' | 'full') and boundary guidance are
#   switches. init='interp', schedule='adaptive', guidance=True is CAKGI's recipe (the same
#   update as impute_pidck); init='noise', schedule='full', guidance=False is plain diffusion
#   (the same update as impute_csdi).
# ════════════════════════════════════════════════════════════════════
def _prep_window(pat, g):
    idx, gl = int(g['start_idx']), int(g['gap_len'])
    ws = max(0, idx - CTX_WIN//5); we = min(len(pat), idx + gl + CTX_WIN//5)
    seg = pat.iloc[ws:we]
    sig = np.stack([(seg['GlucoseCGM'].values - G_MEAN)/G_STD,
                    (seg['HR'].values - H_MEAN)/H_STD], axis=1).astype(np.float32)
    msk = np.isfinite(sig).astype(np.float32)
    return np.nan_to_num(sig, nan=0.).astype(np.float32), msk, idx - ws, gl

def _boundary_weights(L, li, gl):
    bnd = torch.zeros(1, L, NF, device=DEVICE)
    gidx = np.arange(li, li + gl)
    for bi in range(min(2, len(gidx))):
        bnd[:, gidx[bi], :] = 0.8 - bi*0.3
        bnd[:, gidx[-(bi+1)], :] = 0.8 - bi*0.3
    return bnd

def _reverse(eps_fn, alpha_bar, xi, mt, prior, T_eff, init, guidance, bnd):
    """Same update as impute_pidck when init='interp', guidance=True, adaptive T;
    same as impute_csdi / impute_csdi_nc when init='noise', guidance=False, T=T_STEPS."""
    chunks = []
    with torch.inference_mode():
        for nb in chain_chunks(N_SAMPLES, CHAIN_BATCH_SIZE):
            xi_b = xi.expand(nb, -1, -1).contiguous(); mt_b = mt.expand(nb, -1, -1).contiguous()
            pr_b = prior.expand(nb, -1, -1).contiguous(); bnd_b = bnd.expand(nb, -1, -1)
            if init == 'interp':
                ns = alpha_bar[T_eff-1].sqrt()
                x = pr_b + (1 - ns)*torch.randn_like(pr_b)
            else:
                x = torch.randn_like(xi_b)
            x = x*(1 - mt_b) + xi_b*mt_b
            for t in reversed(range(1, T_eff + 1)):
                tt  = torch.full((nb,), t, device=DEVICE, dtype=torch.long)
                eps = eps_fn(x, tt, mt_b, xi_b)
                a_t = alpha_bar[t-1]
                a_p = alpha_bar[t-2] if t > 1 else torch.tensor(1., device=DEVICE)
                x0  = ((x - (1-a_t).sqrt()*eps)/a_t.sqrt()).clamp(-3, 3)
                x   = a_p.sqrt()*x0 + (1-a_p).sqrt()*torch.randn_like(x) if t > 1 else x0
                if guidance and t > T_eff//2:
                    x = x - bnd_b*(t/T_eff)*0.15*(x - pr_b)
                x = x*(1 - mt_b) + xi_b*mt_b
            chunks.append(x.cpu())
    return torch.cat(chunks, 0).numpy()          # (N, L, NF)

def _summarise(sn, sig, msk, li, gl):
    bi = select_best_chain(sn, sig, msk, target_start=li, target_len=gl)
    best, med, sl = sn[bi], np.median(sn, axis=0), slice(li, li + gl)
    return {'glucose': (best[sl, 0]*G_STD + G_MEAN).tolist(),
            'hr':      (best[sl, 1]*H_STD + H_MEAN).tolist(),
            'glucose_med': (med[sl, 0]*G_STD + G_MEAN).tolist(),
            'hr_med':      (med[sl, 1]*H_STD + H_MEAN).tolist(),
            'glucose_samples': (sn[:, sl, 0]*G_STD + G_MEAN).astype(np.float32),
            'hr_samples':      (sn[:, sl, 1]*H_STD + H_MEAN).astype(np.float32)}

def make_diffusion_imputer(eps_factory, alpha_bar, init='interp', schedule='adaptive', guidance=True):
    # A short (adaptive) schedule only makes sense when starting from an informative init.
    assert not (init == 'noise' and schedule == 'adaptive'), "noise init requires schedule='full'"
    def _imp(masked_df, gap_info):
        res = {}
        for g in gap_info:
            pat = _patient(masked_df, g)
            sig, msk, li, gl = _prep_window(pat, g); L = len(sig)
            if L > MAX_L:
                res[id(g)] = {'glucose': [np.nan]*gl, 'hr': [np.nan]*gl}; continue
            xi    = torch.tensor(sig).unsqueeze(0).to(DEVICE)
            mt    = torch.tensor(msk).unsqueeze(0).to(DEVICE)
            prior = torch.tensor(PRIOR_FN(sig, msk), dtype=torch.float32).unsqueeze(0).to(DEVICE)
            T_eff = adaptive_T(gl) if schedule == 'adaptive' else T_STEPS
            sn = _reverse(eps_factory(pat, g), alpha_bar, xi, mt, prior, T_eff,
                          init, guidance, _boundary_weights(L, li, gl))
            res[id(g)] = _summarise(sn, sig, msk, li, gl)
        return res
    return _imp


# ── eps-network adapters: all take / return (B, L, NF) ──
def eps_pidck(model, zero_ctx=False, no_hist=False):
    def factory(pat, g):
        if zero_ctx:
            cp = torch.zeros(1, PRE_DIM_FULL, device=DEVICE)
            cq = torch.zeros(1, POST_DIM_FULL, device=DEVICE)
        else:
            pv, qv = build_pidck_ctx(pat, g['ts_start'], g['ts_end'], g['gap_len']*5,
                                     pm_arr, ps_arr, qm_arr, qs_arr)
            cp = torch.tensor(pv, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            cq = torch.tensor(qv, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            if no_hist:
                cp = cp * HIST_KEEP
        # MPS crashes on zero-stride (expanded) inputs to Linear/LayerNorm, so materialise
        # one real copy per chain batch size, exactly as impute_pidck did with .contiguous().
        rep = {}
        def eps(x, tt, m, xi):
            B = x.shape[0]
            if B not in rep:
                rep[B] = (cp.repeat(B, 1).contiguous(), cq.repeat(B, 1).contiguous())
            return model(x, tt, m, *rep[B])
        return eps
    return factory

def eps_csdi(model):
    def eps(x, tt, m, xi):           # CSDI is channels-first and takes [x_noisy | x_obs | mask]
        xc, mc, xic = x.transpose(1, 2), m.transpose(1, 2), xi.transpose(1, 2)
        return model(xc*(1 - mc), tt, xic*mc, mc).transpose(1, 2)
    return lambda pat, g: eps

print("Diffusion sampler ready.")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 19 — Retrained ablation models (TRAIN patients only, 75 epochs each)
#   Same data, masking, optimiser, schedule and epochs as CAKGI (train_pidck, Cell 10);
#   exactly one ingredient changes per variant.
# ════════════════════════════════════════════════════════════════════
VARIANTS = {   # arm -> (checkpoint, train_pidck keyword arguments)
    'pidck_noctx'     : (_ck('pidck_v4_noctx'),     dict(zero_ctx=True)),
    'pidck_nohist'    : (_ck('pidck_v4_nohist'),    dict(no_hist=True)),
    'pidck_pinpw'     : (_ck('pidck_v4_pinpw'),     dict(pin_pw=True)),
    'pidck_no_bndloss': (_ck('pidck_v4_nobndloss'), dict(bnd_w=0.0)),
    'pidck_no_cauchy' : (_ck('pidck_v4_nocauchy'),  dict(gamma=0.0)),
    'pidck_reseed'    : (_ck('pidck_v4_seed2027'),  dict(seed=SPLIT_SEED + 1)),
}
VARIANT_MODELS = {arm: train_pidck(train_df, ckpt, tag=arm, **kw) for arm, (ckpt, kw) in VARIANTS.items()}

# CSDI-adapted retrained with CAKGI's boundary-aware loss and masking regime (75 epochs).
model_csdi_bnd = train_csdi(train_df, CSDI_BND_MODEL, boundary_loss=True, tag="CSDI-adapted + boundary loss")
print("Ablation models ready.")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 20 — Run every Part B arm on the SAME 3 × N_GAPS gaps (resumable)
# ════════════════════════════════════════════════════════════════════
def _cakgi_arm(model, zero_ctx=False, no_hist=False, init='interp', schedule='adaptive', guidance=True):
    return make_diffusion_imputer(eps_pidck(model, zero_ctx, no_hist), model.alpha_bar,
                                  init, schedule, guidance)

ABL_ARMS = {   # cheap arms first
    'tod_raw'         : lambda d, gi: impute_tod(d, gi, anchored=False),
    'ctx_ridge_cakgi' : make_ridge_imputer('cakgi'),
    'pidck_ref'       : _cakgi_arm(model_pidck),                                      # CAKGI (reference)
    'pidck_noise'     : _cakgi_arm(model_pidck, init='noise', schedule='full', guidance=False),
    'pidck_noctx'     : _cakgi_arm(VARIANT_MODELS['pidck_noctx'], zero_ctx=True),
    'pidck_nohist'    : _cakgi_arm(VARIANT_MODELS['pidck_nohist'], no_hist=True),
    'pidck_pinpw'     : _cakgi_arm(VARIANT_MODELS['pidck_pinpw']),
    'pidck_no_bndloss': _cakgi_arm(VARIANT_MODELS['pidck_no_bndloss']),
    'pidck_no_cauchy' : _cakgi_arm(VARIANT_MODELS['pidck_no_cauchy']),
    'pidck_reseed'    : _cakgi_arm(VARIANT_MODELS['pidck_reseed']),
    'csdi_interp'     : make_diffusion_imputer(eps_csdi(model_csdi), model_csdi.alpha_bar,
                                               'interp', 'adaptive', True),
    'csdi_bnd'        : make_diffusion_imputer(eps_csdi(model_csdi_bnd), model_csdi_bnd.alpha_bar,
                                               'interp', 'adaptive', True),
}
abl_df = run_arms(ABL_ARMS, ABL_RESULTS, trials=range(1, N_TRIALS + 1))
if TOD_FALLBACK['total']:
    print(f"tod_raw: {TOD_FALLBACK['n']/TOD_FALLBACK['total']:.1%} of target steps had no prior-day "
          f"history and fell back to linear interpolation")

# ── Part A + Part B in one file (input of Cells 13, 15 and 22) ──
main_df = pd.read_csv(MAIN_RESULTS); main_df['pid'] = main_df['pid'].astype(str)
all_df = pd.concat([main_df, abl_df[~abl_df['method'].isin(set(main_df['method']))]], ignore_index=True)
all_df.to_csv(ALL_RESULTS, index=False)
print(f"Saved {len(all_df):,} rows ({all_df['method'].nunique()} methods/arms) -> {ALL_RESULTS}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 21 — Ablation table (Table: ablations; RMSE, median of chains)
#   Δ = configuration − CAKGI (pidck_ref), with the patient-level 95 % CI from Cell 15's
#   bootstrap. Needs Cell 1, Cell 15 (paired_patient_bootstrap) and ALL_RESULTS.
# ════════════════════════════════════════════════════════════════════
A = pd.read_csv(ALL_RESULTS)
REF = 'pidck_ref'
ABL_ROWS = [   # (arm, label); None = \midrule
    ('pidck_noctx',      r'$-$ all context (retrained)'),
    ('pidck_nohist',     r'$-$ 1/3/7-day history only (retrained)'),
    ('pidck_pinpw',      r'$-$ learned position weights (retrained)'),
    ('pidck_no_bndloss', r'$-$ boundary loss (retrained)'),
    ('pidck_no_cauchy',  r'$-$ Cauchy regulariser (retrained)'),
    ('pidck_noise',      r'$\pm$ interp.\ init / adaptive schedule / boundary guidance'),
    ('pidck_reseed',     r'Retraining noise floor (CAKGI, second seed)'),
    None,
    ('csdi_interp',      r'CSDI-adapted + CAKGI inference recipe'),
    ('csdi_bnd',         r'CSDI-adapted + boundary loss + CAKGI inference recipe'),
    None,
    ('tod_raw',          r'Time-of-day profile (own, previous 7 days)'),
    ('ctx_ridge_cakgi',  r'Context ridge (CAKGI context vector)'),
]

def _rmse(method, signal):
    return A[(A['method'] == method) & (A['agg'] == 'median') & (A['signal'] == signal)]['rmse'].mean()

stats = {}
for row in ABL_ROWS:
    if row is None or row[0] not in set(A['method']):
        continue
    for signal in ['glucose', 'hr']:          # one bootstrap per arm × signal (seed 0), as in the paper
        for r in paired_patient_bootstrap(A, row[0], REF, agg='median', signals=(signal,), metrics=('rmse',)):
            stats[(row[0], signal)] = r

def _d(v, nd=1):
    s = f"{v:+.{nd}f}" if abs(round(v, nd)) > 0 else f"{0:.{nd}f}"
    return s

def _cell(arm, signal):
    r = stats.get((arm, signal))
    if r is None:
        return r"N/A & N/A"
    nd = 2 if abs(r['mean_diff']) < 0.1 else 1
    return (f"${_rmse(arm, signal):.2f}$ & ${_d(r['mean_diff'], nd)}$ "
            f"[${r['pat_ci_lo']:.{nd}f}$,\\,${r['pat_ci_hi']:.{nd}f}$]")

n_pat = {s: A[(A['method'] == REF) & (A['signal'] == s) & A['rmse'].notna()]['pid'].nunique() for s in ['glucose', 'hr']}
main_ck = {s: _rmse('pidck', s) for s in ['glucose', 'hr']}
L = [r"\begin{table*}[t]", r"\centering", r"\scriptsize",
     r"\caption{Component ablations, reviewer-requested runs and context baselines (median of chains). "
     r"$\Delta$ = configuration $-$ CAKGI (RMSE) with patient-level 95\% CI "
     rf"({n_pat['glucose']} held-out patients for glucose, {n_pat['hr']} with HR ground truth). "
     r"All learned models trained for 75 epochs.}",
     r"\label{tab:ablation}", r"\setlength{\tabcolsep}{4pt}", r"\begin{tabular}{@{}lcccc@{}}", r"\toprule",
     r"& \multicolumn{2}{c}{\textbf{Glucose} (mg/dL)} & \multicolumn{2}{c}{\textbf{HR} (bpm)} \\",
     r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}",
     r"\textbf{Configuration} & \textbf{RMSE} & \textbf{$\Delta$ [95\% CI]} & \textbf{RMSE} & \textbf{$\Delta$ [95\% CI]} \\",
     r"\midrule", "CAKGI$^{*}$",
     f"& ${_rmse(REF, 'glucose'):.2f}$ & ---", f"& ${_rmse(REF, 'hr'):.2f}$ & --- \\\\", r"\midrule"]
for row in ABL_ROWS:
    if row is None:
        L.append(r"\midrule"); continue
    arm, label = row
    L += [label, f"& {_cell(arm, 'glucose')}", f"& {_cell(arm, 'hr')} \\\\"]
L += [r"\bottomrule", r"\end{tabular}\\[0.3em]",
      rf"\footnotesize $^{{*}}$Ablation reference run; it differs from the CAKGI result in Table~\ref{{tab:mc_main}} "
      rf"({main_ck['glucose']:.2f}\,mg/dL, {main_ck['hr']:.2f}\,bpm) by at most "
      rf"{max(abs(_rmse(REF, s) - main_ck[s]) for s in main_ck):.2f}.",
      r"\end{table*}"]
latex_abl = '\n'.join(L)
open(f"{RESULTS_DIR}/table_ablation.tex", 'w').write(latex_abl)
print(latex_abl + f"\n\nSaved -> {RESULTS_DIR}/table_ablation.tex")

pb_abl = pd.DataFrame([r for row in ABL_ROWS if row is not None and row[0] in set(A['method'])
                       for r in paired_patient_bootstrap(A, row[0], REF, agg='median')])   # all metrics
pb_abl.to_csv(f"{RESULTS_DIR}/paired_bootstrap_ablation_vs_ref.csv", index=False)
with pd.option_context('display.width', 250, 'display.max_rows', 300):
    print(pb_abl[pb_abl['metric'] == 'rmse'][['ours', 'signal', 'mean_diff', 'pat_ci_lo', 'pat_ci_hi',
                                               'patients_better']].round(3).to_string(index=False))
print("(here mean_diff > 0 = the configuration is WORSE than CAKGI; patients_better counts patients "
      "for whom the configuration has the lower error)")